In [ ]:
%pip install -q datasets

from datasets import load_dataset

dataset = load_dataset("lhoestq/conll2003")
train = dataset["train"]
validation = dataset["validation"]
test = dataset["test"]

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


Загрузил CoNLL-2003. Из train возьму предложения для обучения, а validation и test использую для оценки

In [ ]:
%pip install -q openai

import json
from pathlib import Path

from google.colab import drive, userdata
from openai import OpenAI

drive.mount("/content/drive")

runs_dir = Path("/content/drive/MyDrive/llm-ner-conll2003/runs")
runs_dir.mkdir(parents=True, exist_ok=True)

client = OpenAI(
    base_url="https://litellm.ispras.ru/v1",
    api_key=userdata.get("LITELLM_API_KEY"),
)

Mounted at /content/drive


Подключил LiteLLM, чтобы получить метки для train, и Google Drive, чтобы сохранить их для обучения маленькой модели

In [ ]:
tag_names = [
    "O", "B-PER", "I-PER", "B-ORG", "I-ORG",
    "B-LOC", "I-LOC", "B-MISC", "I-MISC",
]

Взял порядок меток из описания CoNLL-2003. Числовой код метки используется как индекс в этом списке

Переношу функции из 1conll.ipynb, чтобы размечать train тем же промптом и с той же проверкой ответов, которые использовал для итогового прогона на первом этапе

In [ ]:
def make_prompt(tokens):
    return f"""
Label the named entities in this English text using BIO tags.

Entity types:
- PER: person
- ORG: organization
- LOC: location
- MISC: other named entities, including nationality adjectives

Use B-TYPE for the first token of an entity, I-TYPE for its continuation,
and O for tokens outside any entity.

Return only a JSON array of strings, with exactly one tag per input token.
Keep the original token order.

Tokens: {json.dumps(tokens)}
""".strip()

make_prompt - принимает список токенов и собирает из инструкции и токенов одну строку для запроса к GPT. Токены записываю в JSON, чтобы явно передать их границы и порядок. Прошу модель вернуть по одной BIO-метке на токен

In [ ]:
def parse_llm_tags(answer, token_count, allowed_tags):
    tags = json.loads(answer)

    if not isinstance(tags, list):
        raise ValueError("Ответ модели должен быть JSON-массивом меток")
    if len(tags) != token_count:
        raise ValueError("Число меток не совпадает с числом токенов")

    previous_tag = "O"

    for tag in tags:
        if tag not in allowed_tags:
            raise ValueError(f"Неизвестная метка: {tag}")

        if tag.startswith("I-"):
            begin_tag = f"B-{tag[2:]}"
            if previous_tag not in (begin_tag, tag):
                raise ValueError(f"Метка {tag} стоит после {previous_tag}")

        previous_tag = tag

    return tags

parse_llm_tags - читает строку ответа как JSON и возвращает список меток, если он прошёл проверку. Проверяю, что меток столько же, сколько токенов, что каждая метка есть в tag_names и что продолжение сущности стоит после начала или продолжения того же типа. Например, I-LOC после O считаю ошибкой. Если модель поставила LOC вместо ORG, ответ может пройти эту проверку, а ошибку в типе покажет сравнение с эталоном

In [ ]:
def parse_model_tags(answer, token_count, allowed_tags):
    if not isinstance(answer, str):
        raise ValueError("Модель не вернула строку")

    lines = answer.strip().splitlines()
    if (
        len(lines) >= 2
        and lines[0].strip().lower() in ("```json", "```")
        and lines[-1].strip() == "```"
    ):
        answer = "\n".join(lines[1:-1])

    return parse_llm_tags(answer, token_count, allowed_tags)

parse_model_tags - подготавливает ответ перед проверкой BIO. На первом этапе Gemma иногда присылала массив внутри блока кода Markdown, и json.loads не мог прочитать его целиком. Если в ответе есть открывающая и закрывающая строки такого блока, убираю их и передаю содержимое в parse_llm_tags

In [ ]:
def save_answer(run_path, result):
    with run_path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(result, ensure_ascii=False) + "\n")

save_answer - записывает результат одного запроса отдельной строкой в файл JSONL. Открываю файл в режиме добавления, поэтому каждый новый ответ дописывается в конец. Сохраняю номер записи и исходный ответ, чтобы потом найти нужное предложение в train, разобрать ошибки и продолжить разметку после перезапуска Colab

In [ ]:
pilot_path = runs_dir / "train_pilot_01.jsonl"
teacher_model = "openai/gpt-oss-120b"
saved_indices = set()

if pilot_path.exists():
    with pilot_path.open(encoding="utf-8") as file:
        for line in file:
            saved_indices.add(json.loads(line)["row_index"])

for row_index in range(5):
    if row_index in saved_indices:
        print(f"Запись {row_index}: ответ уже сохранён")
        continue

    row_tokens = train[row_index]["tokens"]
    response = client.chat.completions.create(
        model=teacher_model,
        messages=[{"role": "user", "content": make_prompt(row_tokens)}],
    )
    answer = response.choices[0].message.content

    predicted_tags = None
    format_error = None

    try:
        predicted_tags = parse_model_tags(answer, len(row_tokens), tag_names)
    except (ValueError, TypeError) as error:
        format_error = str(error)

    save_answer(pilot_path, {
        "row_index": row_index,
        "tokens": row_tokens,
        "model_id": teacher_model,
        "answer": answer,
        "predicted_tags": predicted_tags,
        "format_error": format_error,
    })

    print(f"Запись {row_index}: {format_error or 'метки сохранены'}")

print("Файл:", pilot_path)

Запись 0: ответ уже сохранён
Запись 1: ответ уже сохранён
Запись 2: ответ уже сохранён
Запись 3: ответ уже сохранён
Запись 4: ответ уже сохранён
Файл: /content/drive/MyDrive/llm-ner-conll2003/runs/train_pilot_01.jsonl


Разметил первые пять записей из train, чтобы проверить, как сохраняются ответы GPT перед обучением. Для каждой записи сохранил её номер, исходные токены и ответ модели в поле answer. Если ответ прошёл проверку, записал список меток в predicted_tags. Если проверка не прошла, оставил там None и сохранил причину в format_error

In [ ]:
pilot_pairs = []

with pilot_path.open(encoding="utf-8") as file:
    for line in file:
        saved = json.loads(line)
        row_index = saved["row_index"]
        record = train[row_index]

        if saved["tokens"] != record["tokens"]:
            raise ValueError(f"Не совпали токены в записи {row_index}")

        if saved["predicted_tags"] is None:
            print(f"Запись {row_index}: пока нет пригодных меток GPT")
            continue

        gold_tags = [tag_names[tag_id] for tag_id in record["ner_tags"]]

        pilot_pairs.append({
            "row_index": row_index,
            "tokens": record["tokens"],
            "llm_tags": saved["predicted_tags"],
            "gold_tags": gold_tags,
        })

print("Записей для сравнения:", len(pilot_pairs))
print(pilot_pairs[0])

Запись 3: пока нет пригодных меток GPT
Запись 4: пока нет пригодных меток GPT
Записей для сравнения: 3
{'row_index': 0, 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'llm_tags': ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O'], 'gold_tags': ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']}


По номеру записи нашёл исходное предложение в train и проверил, что токены совпадают с сохранёнными. Для трёх пригодных ответов собрал метки GPT и эталонные метки того же предложения. Две записи с ошибкой длины пока пропустил, чтобы оба набора для обучения содержали одни и те же предложения

In [ ]:
import random


def label_train_rows(row_indices, output_path):
    saved_indices = set()

    if output_path.exists():
        with output_path.open(encoding="utf-8") as file:
            for line in file:
                saved_indices.add(json.loads(line)["row_index"])

    for row_index in row_indices:
        if row_index in saved_indices:
            continue

        row_tokens = train[row_index]["tokens"]
        response = client.chat.completions.create(
            model=teacher_model,
            messages=[{"role": "user", "content": make_prompt(row_tokens)}],
        )
        answer = response.choices[0].message.content

        predicted_tags = None
        format_error = None
        try:
            predicted_tags = parse_model_tags(answer, len(row_tokens), tag_names)
        except (ValueError, TypeError) as error:
            format_error = str(error)

        save_answer(output_path, {
            "row_index": row_index,
            "tokens": row_tokens,
            "model_id": teacher_model,
            "answer": answer,
            "predicted_tags": predicted_tags,
            "format_error": format_error,
        })

        saved_indices.add(row_index)


audit_indices = random.Random(42).sample(range(5, len(train)), 50)
audit_path = runs_dir / "train_audit_01.jsonl"

label_train_rows(audit_indices, audit_path)
print("Ответы сохранены в", audit_path)

Ответы сохранены в /content/drive/MyDrive/llm-ner-conll2003/runs/train_audit_01.jsonl


label_train_rows - отправляет записи из train в GPT и сохраняет текст каждого ответа. Если список BIO-меток прошёл проверку длины и правил BIO, функция сохраняет этот список. При ошибке записывает её причину

Выбрал 50 случайных записей с seed 42, исключив первые пять, на которых проверял код. Сохранил ответы в train_audit_01.jsonl. По этому файлу проверю, какая доля ответов пригодна для обучения и насколько метки GPT совпадают с эталоном

Переношу функции из 1conll.ipynb, чтобы сравнить метки GPT с эталоном по тем же правилам, что на первом этапе: сущность засчитывается, если совпали обе границы и тип

In [ ]:
def bio_to_spans(tags):
    spans = []
    i = 0

    while i < len(tags):
        tag = tags[i]
        if tag == "O":
            i += 1
            continue

        start = i
        entity_type = tag[2:]
        i += 1

        while i < len(tags) and tags[i] == f"I-{entity_type}":
            i += 1

        spans.append((start, i, entity_type))

    return spans

bio_to_spans - функция, которая собирает целые сущности из BIO-меток, чтобы потом сравнить их границы и типы

In [ ]:
def count_matches(gold_tags, predicted_tags):
    if len(gold_tags) != len(predicted_tags):
        raise ValueError("Число меток в двух разметках должно совпадать.")

    gold = set(bio_to_spans(gold_tags))
    predicted = set(bio_to_spans(predicted_tags))

    tp = len(gold & predicted)
    fp = len(predicted - gold)
    fn = len(gold - predicted)

    return tp, fp, fn


def scores_from_counts(tp, fp, fn):
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    return precision, recall, f1

В count_matches сравниваю ответ с эталоном и считаю, сколько сущностей нашли правильно (tp - true positive), сколько выделили лишних (fp - false positive) и сколько пропустили (fn - false negative)
В scores_from_counts считаю precision, recall и F1

In [ ]:
with audit_path.open(encoding="utf-8") as file:
    audit_results = [json.loads(line) for line in file]

audit_tp, audit_fp, audit_fn = 0, 0, 0
failed_count = 0
failed_gold_count = 0

for result in audit_results:
    record = train[result["row_index"]]
    gold_tags = [tag_names[tag_id] for tag_id in record["ner_tags"]]
    predicted_tags = result["predicted_tags"]

    if predicted_tags is None:
        failed_count += 1
        failed_gold_count += len(bio_to_spans(gold_tags))
        continue

    tp, fp, fn = count_matches(gold_tags, predicted_tags)
    audit_tp += tp
    audit_fp += fp
    audit_fn += fn

_, _, valid_f1 = scores_from_counts(audit_tp, audit_fp, audit_fn)
_, _, full_f1 = scores_from_counts(
    audit_tp, audit_fp, audit_fn + failed_gold_count
)

print(f"Пригодных ответов: {len(audit_results) - failed_count}/{len(audit_results)}")
print(f"Micro-F1 на пригодных ответах: {valid_f1:.2%}")
print(f"Micro-F1 на всех 50 записях: {full_f1:.2%}")
print("Эталонных сущностей в записях без пригодного ответа:", failed_gold_count)

Пригодных ответов: 44/50
Micro-F1 на пригодных ответах: 80.67%
Micro-F1 на всех 50 записях: 68.57%
Эталонных сущностей в записях без пригодного ответа: 21


С первой попытки проверку прошли 44 ответа из 50. На них micro-F1 получился 80,67%. В шести записях без пригодных меток была 21 эталонная сущность. Если учесть их как пропуски, micro-F1 по всей выборке составляет 68,57%

Для шести записей, по которым первый ответ GPT не прошёл проверку, отправлю по одному повторному запросу с тем же промптом. Проверю, для скольких из них удастся получить пригодные метки для обучения

In [ ]:
retry_indices = [
    result["row_index"]
    for result in audit_results
    if result["predicted_tags"] is None
]
retry_path = runs_dir / "train_audit_retry_01.jsonl"

label_train_rows(retry_indices, retry_path)

with retry_path.open(encoding="utf-8") as file:
    retry_results = [json.loads(line) for line in file]

recovered_count = sum(
    result["predicted_tags"] is not None
    for result in retry_results
)

print(f"После повтора получили пригодные метки: {recovered_count}/{len(retry_indices)}")

После повтора получили пригодные метки: 5/6


Повторил запрос для шести записей, по которым первый ответ не прошёл проверку. Пять новых ответов прошли проверку, поэтому теперь есть пригодные метки для 49 из 50 записей. Дальше сравню эти пять ответов с эталоном и пересчитаю micro-F1

In [ ]:
retry_tp, retry_fp, retry_fn = audit_tp, audit_fp, audit_fn
remaining_failed_gold_count = failed_gold_count

for result in retry_results:
    predicted_tags = result["predicted_tags"]
    if predicted_tags is None:
        continue

    record = train[result["row_index"]]
    gold_tags = [tag_names[tag_id] for tag_id in record["ner_tags"]]

    remaining_failed_gold_count -= len(bio_to_spans(gold_tags))
    tp, fp, fn = count_matches(gold_tags, predicted_tags)
    retry_tp += tp
    retry_fp += fp
    retry_fn += fn

_, _, retry_valid_f1 = scores_from_counts(retry_tp, retry_fp, retry_fn)
_, _, retry_full_f1 = scores_from_counts(
    retry_tp, retry_fp, retry_fn + remaining_failed_gold_count
)

print(f"Micro-F1 на пригодных ответах после повтора: {retry_valid_f1:.2%}")
print(f"Micro-F1 на всех 50 записях после повтора: {retry_full_f1:.2%}")

Micro-F1 на пригодных ответах после повтора: 81.25%
Micro-F1 на всех 50 записях после повтора: 80.25%


Для каждой из шести записей, где первый ответ GPT не прошёл проверку формата и правил BIO, отправил по одному повторному запросу с тем же промптом. При пересчёте метрик использовал первый ответ, если он прошёл проверку, и повторный, если проверку прошёл только он

На записях с пригодными ответами micro-F1 составил 81,25%. На всех 50 записях micro-F1 вырос с 68,57% до 80,25%. Эталонные сущности из записи, для которой обе попытки не прошли проверку, засчитал как пропущенные

При разметке train буду один раз повторять запрос для записей, где первый ответ не прошёл проверку формата и правил BIO

Начинаю собирать разметку для обучения. Перемешиваю индексы train с фиксированным seed и сначала беру 100 записей

Если первый ответ GPT не прошёл проверку формата и правил BIO, отправляю тот же запрос ещё один раз. Ответы двух попыток сохраняю в отдельных файлах

In [ ]:
train_order = list(range(len(train)))
random.Random(42).shuffle(train_order)
train_indices = train_order[:100]

train_first_path = runs_dir / "train_stage2_first.jsonl"
train_retry_path = runs_dir / "train_stage2_retry.jsonl"

label_train_rows(train_indices, train_first_path)

with train_first_path.open(encoding="utf-8") as file:
    train_first_results = [json.loads(line) for line in file]

train_retry_indices = [
    result["row_index"]
    for result in train_first_results
    if result["predicted_tags"] is None
]

print(f"Ответов первой попытки в файле: {len(train_first_results)}")
print(f"Записей для повторной попытки: {len(train_retry_indices)}")

label_train_rows(train_retry_indices, train_retry_path)

Ответов первой попытки в файле: 100
Записей для повторной попытки: 8


In [ ]:
first_tags_by_index = {
    result["row_index"]: result["predicted_tags"]
    for result in train_first_results
}

retry_tags_by_index = {}
if train_retry_indices:
    with train_retry_path.open(encoding="utf-8") as file:
        for line in file:
            result = json.loads(line)
            retry_tags_by_index[result["row_index"]] = result["predicted_tags"]

train_pairs = []
valid_retry_count = 0

for row_index in train_indices:
    gpt_tags = first_tags_by_index[row_index]

    if gpt_tags is None:
        gpt_tags = retry_tags_by_index[row_index]
        if gpt_tags is None:
            continue
        valid_retry_count += 1

    record = train[row_index]
    train_pairs.append({
        "row_index": row_index,
        "tokens": record["tokens"],
        "gpt_tags": gpt_tags,
        "gold_tags": [tag_names[tag_id] for tag_id in record["ner_tags"]],
    })

print(f"Пригодные метки после повтора: {valid_retry_count}/{len(train_retry_indices)}")
print(f"Записей с двумя версиями разметки: {len(train_pairs)}/{len(train_indices)}")

Пригодные метки после повтора: 5/8
Записей с двумя версиями разметки: 97/100


Начинаю подготовку данных с DistilBERT. На заголовке с LEICESTERSHIRE из validation посмотрю, как токенизатор делит исходные токены на части и как по word_id можно найти исходный токен для каждой части

In [ ]:
from transformers import AutoTokenizer

student_checkpoint = "distilbert/distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(student_checkpoint, use_fast=True)

tokens = validation[0]["tokens"]
encoded = tokenizer(tokens, is_split_into_words=True)

for subtoken, word_id in zip(encoded.tokens(), encoded.word_ids()):
    original_token = tokens[word_id] if word_id is not None else None
    print(subtoken, word_id, original_token)

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

[CLS] None None
CR 0 CRICKET
##IC 0 CRICKET
##KE 0 CRICKET
##T 0 CRICKET
- 1 -
L 2 LEICESTERSHIRE
##EI 2 LEICESTERSHIRE
##CE 2 LEICESTERSHIRE
##ST 2 LEICESTERSHIRE
##ER 2 LEICESTERSHIRE
##S 2 LEICESTERSHIRE
##H 2 LEICESTERSHIRE
##IR 2 LEICESTERSHIRE
##E 2 LEICESTERSHIRE
T 3 TAKE
##A 3 TAKE
##KE 3 TAKE
O 4 OVER
##VE 4 OVER
##R 4 OVER
AT 5 AT
TO 6 TOP
##P 6 TOP
A 7 AFTER
##FT 7 AFTER
##ER 7 AFTER
IN 8 INNINGS
##NI 8 INNINGS
##NG 8 INNINGS
##S 8 INNINGS
VI 9 VICTORY
##CT 9 VICTORY
##OR 9 VICTORY
##Y 9 VICTORY
. 10 .
[SEP] None None


transformers — библиотека Hugging Face для работы с моделями и токенизаторами. Токенизатор разбивает текст на части и переводит их в числа, которые можно передать модели. AutoTokenizer по идентификатору модели подбирает подходящий токенизатор

Чекпоинт — сохранённое состояние модели, в частности её обученные веса. В student_checkpoint храню идентификатор distilbert/distilbert-base-cased. По нему можно найти на Hugging Face и веса модели, и файлы её токенизатора. В этой ячейке AutoTokenizer.from_pretrained загружает словарь и настройки токенизатора. Веса DistilBERT загружу позже, когда буду создавать модель для обучения

use_fast=True просит использовать быструю реализацию токенизатора, если она доступна. Реализация — это код, который выполняет токенизацию. Быстрая версия использует библиотеку на Rust.

is_split_into_words=True говорит токенизатору воспринимать элементы списка как исходные токены одного текста. Без этого параметра список строк будет обработан как несколько отдельных текстов. Например, ["New", "York"] с этим параметром — один текст из двух исходных токенов с индексами 0 и 1. Без параметра это два отдельных текста, и в каждом нумерация начнётся с 0

Теперь использую word_id, чтобы привязать BIO-метку исходного токена к его первой части. Для остальных частей и служебных токенов поставлю -100, чтобы пропустить эти позиции при расчёте ошибки обучения

In [ ]:
tag_to_id = {tag: index for index, tag in enumerate(tag_names)}


def align_bio_labels(word_ids, tags):
    labels = []
    previous_word_id = None

    for word_id in word_ids:
        if word_id is None or word_id == previous_word_id:
            labels.append(-100)
        else:
            labels.append(tag_to_id[tags[word_id]])

        previous_word_id = word_id

    return labels


example = train_pairs[0]
encoded = tokenizer(example["tokens"], is_split_into_words=True)
word_ids = encoded.word_ids()

gpt_labels = align_bio_labels(word_ids, example["gpt_tags"])
gold_labels = align_bio_labels(word_ids, example["gold_tags"])

print("Подтокен | Метка GPT | Эталон")
for subtoken, gpt_id, gold_id in zip(encoded.tokens(), gpt_labels, gold_labels):
    gpt_tag = tag_names[gpt_id] if gpt_id != -100 else "—"
    gold_tag = tag_names[gold_id] if gold_id != -100 else "—"
    print(f"{subtoken} | {gpt_tag} | {gold_tag}")

Подтокен | Метка GPT | Эталон
[CLS] | — | —
A | B-PER | B-PER
. | — | —
de | — | —
Silva | I-PER | I-PER
not | O | O
out | O | O
49 | O | O
[SEP] | — | —


align_bio_labels — подготавливает метки для обучения модели. Возвращает список числовых меток для обучения модели. В этом списке по одному значению на каждую позицию после токенизации. Первая часть исходного токена получает числовой код его BIO-метки, а остальные части и служебные токены получают -100, чтобы пропустить их при расчёте ошибки обучения

Функция принимает word_ids и tags. Из word_ids беру индекс исходного токена, по этому индексу нахожу его BIO-метку в tags и через tag_to_id перевожу её в числовой код

Например, если LEICESTERSHIRE стоит на позиции 2 и разбился на три части, в word_ids будет три значения 2 подряд. Для первой части функция берёт tags[2], например B-ORG, и записывает числовой код этой метки. Для следующих двух частей и для служебных токенов записывает -100

Для каждого из 97 предложений беру исходные токены и две готовые разметки — GPT и эталонную. Токенизирую предложение и привязываю каждую BIO-метку к первой части соответствующего токена. Так получаю два обучающих набора с одинаковыми исходными данными и разными метками

In [ ]:
from datasets import Dataset

gpt_examples = []
gold_examples = []

for pair in train_pairs:
    encoded = tokenizer(pair["tokens"], is_split_into_words=True)
    word_ids = encoded.word_ids()

    gpt_examples.append({
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": align_bio_labels(word_ids, pair["gpt_tags"]),
    })

    gold_examples.append({
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": align_bio_labels(word_ids, pair["gold_tags"]),
    })

gpt_train_dataset = Dataset.from_list(gpt_examples)
gold_train_dataset = Dataset.from_list(gold_examples)

print(gpt_train_dataset[0])
print("Записей с метками GPT:", len(gpt_train_dataset))
print("Записей с эталонными метками:", len(gold_train_dataset))

{'input_ids': [101, 138, 119, 1260, 11211, 1136, 1149, 3927, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 1, -100, -100, 2, 0, 0, 0, -100]}
Записей с метками GPT: 97
Записей с эталонными метками: 97


input_ids — список числовых кодов из словаря токенизатора, по одному коду на каждую позицию после токенизации. По этим кодам модель берёт векторы из таблицы эмбеддингов

encoded.word_ids() — список индексов исходных токенов CoNLL. Если исходный токен разбился на несколько частей, у всех этих частей будет один и тот же индекс. Для добавленных служебных токенов вроде [CLS] и [SEP] стоит None


# Как работает DistilBERT

Пример для всех шагов — порция из **2 предложений по 6 позиций**. У DistilBERT **6 блоков**, в каждом **12 голов внимания**. Размер вектора позиции — **768**.

### 1. `input_ids` → вход первого блока

`input_ids` имеет форму `(2, 6)`. Для каждой позиции модель берёт из одной таблицы **эмбеддинг токена по его ID**, а из другой — **эмбеддинг по номеру позиции**. Оба вектора содержат 768 чисел. Их складывают по координатам.

$$
X = \operatorname{Dropout}\bigl(\operatorname{LayerNorm}(E_{\text{token}} + E_{\text{position}})\bigr)
$$

Форма `X` — **`(2, 6, 768)`**. `LayerNorm` считает среднее $\mu$ и дисперсию $\sigma^2$ **по 768 координатам одного вектора**, а затем преобразует каждую его координату:

$$
\operatorname{LayerNorm}(x)_j =
\gamma_j \frac{x_j-\mu}{\sqrt{\sigma^2+\varepsilon}}+\beta_j.
$$

$\gamma_j$ и $\beta_j$ обучаются. Dropout при обучении обнуляет случайные координаты и делит сохранённые на $1-p$. При оценке передаёт значения как есть.

### 2. Три линейных слоя создают Query, Key и Value

Каждый слой получает **весь** входной вектор $x$ из 768 чисел:

$$
q=W_Qx+b_Q,\qquad k=W_Kx+b_K,\qquad v=W_Vx+b_V.
$$

Все три матрицы имеют форму `(768, 768)`. На каждой позиции получаются **три вектора по 768 чисел**.

### 3. Одна голова считает выход для каждой позиции

Для позиции $i$ голова сравнивает её `Query` $q_i$ с `Key` $k_j$ **каждой** позиции $j$ того же предложения. Скалярное произведение двух векторов из 64 чисел даёт одну оценку. Делим её **на корень из размерности**, то есть на $\sqrt{64}=8$:

$$
s_{ij}=\frac{q_i\cdot k_j}{8}.
$$

Для одного Query получаем 6 оценок. `attention_mask` исключает позиции `[PAD]` из числа источников информации. Затем `softmax` превращает оценки доступных позиций в веса:

$$
a_{ij}=\frac{e^{s_{ij}}}{\sum_{k\ \text{доступен}} e^{s_{ik}}}.
$$

Сумма весов для одного Query равна 1 **до dropout внимания**. На обучении dropout применяется к этим весам. Затем голова умножает каждый вектор `Value` на его вес и складывает результаты по координатам:

$$
z_i=\sum_j \operatorname{Dropout}(a)_{ij}\,v_j.
$$

Каждый $v_j$ содержит 64 числа, поэтому $z_i$ тоже содержит **64 числа**. Голова вычисляет такой вектор **для каждой из 6 позиций**. Выход одной головы для одного предложения — `(6, 64)`. Выход всех голов для всей порции — **`(2, 12, 6, 64)`**.

### 4. Объединение голов → выходной линейный слой → сложение → LayerNorm

Для каждой позиции соединяем 12 выходов по 64 числа. Получаем `(2, 6, 768)`. **Четвёртый линейный слой внимания**, $W_O$ формы `(768, 768)`, смешивает координаты разных голов:

$$
A_i=W_O[z_i^{(1)};\ldots;z_i^{(12)}]+b_O,
\qquad H=\operatorname{LayerNorm}_1(X+A).
$$

`X + A` — остаточное соединение. Оно складывает **вход блока** и **выход внимания**. `H` имеет форму **`(2, 6, 768)`**.

### 5. FFN → сложение → LayerNorm → выход блока

FFN обрабатывает каждый вектор $h_i$ отдельно. Сначала линейный слой расширяет его **из 768 до 3072 чисел**. К каждому числу применяется `GELU`. Второй линейный слой возвращает размер **768**, затем применяется dropout:

$$
F_i=\operatorname{Dropout}\bigl(W_2\,\operatorname{GELU}(W_1h_i+b_1)+b_2\bigr),
\qquad
\operatorname{GELU}(u)=u\,\Phi(u).
$$

$\Phi$ — функция распределения стандартной нормальной величины. Формы матриц — $W_1$ `(3072, 768)` и $W_2$ `(768, 3072)`.

К выходу FFN прибавляем **вход FFN**, то есть `H`, и применяем второй `LayerNorm`:

$$
X_{\text{out}}=\operatorname{LayerNorm}_2(H+F).
$$

Форма выхода блока — **`(2, 6, 768)`**. Этот тензор становится входом следующего блока, где расчёт повторяется с его собственными весами.

### 6. Выход модели

После шестого блока бэкбон возвращает `last_hidden_state` формы **`(2, 6, 768)`**. В линейном NER-бейзлайне эти векторы проходят через dropout и классификатор **`768 → 9`**:

$$
\text{logits}_i=W_{\text{NER}}\,\operatorname{Dropout}(x_i^{\text{final}})+b_{\text{NER}}.
$$

Выход — **`logits` формы `(2, 6, 9)`**, по девять сырых оценок BIO-меток на позицию. При наличии `labels` модель также возвращает `loss`, рассчитанный по кросс-энтропии на позициях с меткой, отличной от `-100`.


**Шесть блоков идут последовательно**, а **двенадцать голов работают параллельно внутри каждого блока**. Первый блок получает векторы из эмбеддингов токенов и позиций. Второй получает выход первого и вычисляет из него свои Query, Key и Value. Так продолжается до шестого блока. Каждый блок имеет свои обучаемые параметры и передаёт дальше по одному вектору из 768 чисел на каждую позицию.

Для задачи распознавания сущностей поверх DistilBERT добавляю **линейный слой классификации**. Он получает итоговый вектор позиции из 768 чисел и выдаёт **9 оценок** (9 логитов), по одной на каждую BIO-метку. При предсказании выбираем метку с наибольшим логитом среди всех девяти. При обучении передаём все девять логитов и целевую метку в функцию потерь. Для каждого исходного токена CoNLL целевая метка стоит на его первой части. Остальные части и служебные токены имеют в списке `labels` значение `-100`, поэтому отдельная ошибка для них не считается.

## Таблица эмбеддингов и начальные векторы

Токенизатор переводит каждый токен или подтокен в **числовой код из словаря**. Эти коды хранятся в `input_ids`. Каждый код указывает на строку в **таблице эмбеддингов**. Таблица эмбеддингов — обучаемая матрица, в которой каждая строка содержит вектор одного элемента словаря.

У каждой строки таблицы эмбеддингов DistilBERT 768 координат. Векторы размерности 64 появятся позже, внутри отдельных голов внимания.

У модели также есть таблица **позиционных эмбеддингов**. Из неё берётся вектор позиции токена в предложении. Модель складывает вектор токена и вектор позиции по координатам, применяет LayerNorm и, в режиме обучения, dropout. После этого векторы поступают в первый блок Transformer.

В примере допустим, что после этих операций на вход первого блока пришли такие векторы:

```text
Hamburg → [1, 1]
won     → [3, 2]
```

Для одного и того же токена строка в таблице эмбеддингов одна и та же. Позиционный вектор зависит от места токена в предложении, а после блоков внимания представление зависит и от окружающих токенов.


## Как из вектора токена получаются Query, Key и Value

**Self-attention, или самовнимание**, позволяет обновить представление каждого токена с учётом других токенов того же предложения. Чтобы выполнить это вычисление, слой применяет к входному вектору токена три отдельных линейных преобразования. У каждого своя матрица весов и своё смещение:

$$
q = W_Qx + b_Q,\qquad
k = W_Kx + b_K,\qquad
v = W_Vx + b_V
$$

- **Query** — вектор токена, для которого сейчас собираю информацию из контекста
- **Key** — вектор, с которым сравниваю Query, чтобы получить оценку для пары токенов
- **Value** — вектор, который затем умножаю на вес внимания и включаю в сумму


## Скалярное произведение, размерность ключа и softmax

Чтобы обновить представление `Hamburg`, сравниваю **Query `Hamburg` с Key каждого токена**, включая сам `Hamburg`. Для сравнения используется **скалярное произведение**: перемножаю координаты на одинаковых позициях и складываю результаты. Из двух векторов получается одно число.

```text
Query Hamburg · Key Hamburg = [1, 0] · [1, 1] = 1 × 1 + 0 × 1 = 1
Query Hamburg · Key won     = [1, 0] · [3, 2] = 1 × 3 + 0 × 2 = 3
```

Оценка для `Hamburg → won` в этом примере выше из-за выбранных чисел. Это оценка вклада `won` при обновлении `Hamburg` в конкретной голове внимания.

Далее делю оценки на $\sqrt{d_k}$, где $d_k$ — **размерность Key**, то есть количество координат. Вектор `[3, 2]` имеет размерность `2`, поэтому здесь делю на $\sqrt{2}$. Масштабирование сдерживает рост величины скалярных произведений при увеличении размерности. Это помогает избежать слишком резкого распределения после softmax.

**Softmax** преобразует набор оценок в веса с суммой `1`. Для каждой оценки $s_i$ вычисляю $e^{s_i}$ и делю на сумму таких значений для всех учитываемых позиций:

$$
\alpha_i = \frac{e^{s_i}}{\sum_j e^{s_j}}
$$

| Шаг | `Hamburg` | `won` |
|---|---:|---:|
| Скалярное произведение | 1 | 3 |
| Деление на $\sqrt{2}$ | ≈ 0.707 | ≈ 2.121 |
| Возведение $e$ в полученную степень | ≈ 2.03 | ≈ 8.34 |
| Деление на общую сумму | ≈ 0.196 | ≈ 0.804 |

Эти числа — **веса внимания для Query `Hamburg`**. Когда модель обновляет `won`, она берёт Query `won` и вычисляет свой набор весов. В реальном входе `attention_mask` также позволяет исключить позиции заполнения `[PAD]` из этого вычисления.


## Головы внимания и объединение их результатов

**Голова внимания** — одно вычисление внимания со своими проекциями Query, Key и Value. Внутри каждого блока DistilBERT таких голов **12**. Количество голов — настройка архитектуры.

## Остаточное соединение и LayerNorm

После выходного линейного слоя внимания результат складывается по координатам с вектором, который поступил на вход внимания. Такое сложение называется **остаточным соединением**. Оно позволяет передать дальше и прежнее представление токена, и результат внимания.

После сложения применяется **LayerNorm**. Она нормализует координаты **внутри вектора одного токена**. Например, для вектора `[2, 6]` среднее равно `4`, а стандартное отклонение равно `2`. Если опустить маленькую константу $\varepsilon$, которая нужна для устойчивости деления, получаю:

$$
\frac{x_i - \text{среднее}}{\sqrt{\text{дисперсия} + \varepsilon}}
$$
```text
[2, 6] → вычитаем среднее → [−2, 2]
       → делим на стандартное отклонение → [−1, 1]
```

Затем каждая нормализованная координата умножается на свой обучаемый **масштаб** $\gamma$ и получает своё обучаемое **смещение** $\beta$. Например, при масштабе `[2, 3]` и смещении `[0.5, 1]`:

```text
Первая координата:  −1 × 2 + 0.5 = −1.5
Вторая координата:   1 × 3 + 1   =  4
Выход: [−1.5, 4]
```

Среднее и стандартное отклонение вычисляются по текущему вектору. Масштаб и смещение хранятся в модели как обучаемые параметры. LayerNorm нужна, чтобы следующий слой получал векторы с более предсказуемым масштабом. Внутри Transformer результат внимания складывается с входным вектором. Такие сложения и преобразования повторяются в каждом блоке, и величина чисел может меняться от слоя к слою. LayerNorm приводит их к сопоставимому масштабу, что помогает модели обучаться стабильнее. LayerNorm применяется и при обучении, и при оценке.

## FFN — нейросеть внутри блока Transformer

После внимания, остаточного соединения и нормализации вектор поступает в **FFN** (*feed-forward network*, сеть прямого распространения). В нашем блоке FFN состоит из двух линейных слоёв с функцией GELU между ними. **Одна и та же FFN применяется отдельно к каждому токену** в пределах блока. К этому моменту каждый токен уже получил информацию о контексте через внимание.

Первый линейный слой FFN преобразует вектор из **768 координат в 3072**. Его матрица весов содержит **3072 строки по 768 чисел**. Каждая строка даёт одну новую координату. Принцип тот же, что в маленьком примере с двумя входами и тремя выходами:

```text
Вход [2, 3]

Строка весов [1, 0] → 1 × 2 + 0 × 3 = 2
Строка весов [0, 2] → 0 × 2 + 2 × 3 = 6
Строка весов [1, 1] → 1 × 2 + 1 × 3 = 5

Выход [2, 6, 5]
```

После первого линейного слоя к каждой из 3072 координат отдельно применяется **GELU** — нелинейная функция активации. Её можно записать так:

$$
\operatorname{GELU}(x)
= x\Phi(x)
= \frac{x}{2}\left[1+\operatorname{erf}\left(\frac{x}{\sqrt{2}}\right)\right]
$$

Здесь $x$ — одна координата, а $\Phi(x)$ — вероятность того, что величина со стандартным нормальным распределением окажется меньше или равна $x$. **erf** (*error function*, функция ошибок) — математическая функция, которая определяется через интеграл:

$$
\operatorname{erf}(z) = \frac{2}{\sqrt{\pi}} \int_0^z e^{-t^2}\,dt
$$

Через erf выражается функция распределения стандартной нормальной величины:

$$
\Phi(x) = \frac{1}{2}\left[1+\operatorname{erf}\left(\frac{x}{\sqrt{2}}\right)\right]
$$

Например, $\operatorname{erf}(0)=0$, поэтому $\Phi(0)=0.5$ и $\operatorname{GELU}(0)=0 \cdot 0.5=0$ Например, `GELU(−1) ≈ −0.16`, `GELU(0) = 0`, `GELU(1) ≈ 0.84`. В коде это вычисляет готовая функция.

GELU меняет значения координат, сохраняя их количество. Благодаря нелинейности между двумя линейными слоями FFN может выучить преобразование, которое сложнее одного умножения на матрицу с прибавлением смещения. Второй линейный слой возвращает размерность 768:

```text
768 → линейный слой → 3072 → GELU → 3072 → линейный слой → 768
```

После второго линейного слоя FFN применяется dropout. Затем результат складывается с входом FFN через ещё одно остаточное соединение и проходит через LayerNorm.

## Dropout — случайное обнуление при обучении

**Dropout** временно обнуляет случайные компоненты активаций при проходе в режиме обучения. Например, для вектора `[2, 4, 6, 8]` может выпасть маска `[1, 0, 1, 0]`. После умножения по координатам получается `[2, 0, 6, 0]`.

В PyTorch сохранённые значения при обучении дополнительно масштабируются на $1/(1-p)$, где $p$ — вероятность обнуления одной компоненты. При $p=0.5$ это множитель `2`, поэтому выход примера равен `[4, 0, 12, 0]`. При следующем проходе маска может быть другой. Это помогает модели меньше полагаться на отдельные компоненты.

В нашем DistilBERT dropout встроен в модель. В конфигурации чекпоинта `dropout = 0.1` и `attention_dropout = 0.1`. В режиме `model.train()` он срабатывает при проходе автоматически, а в режиме `model.eval()` отключается. Dropout на весах внимания применяется после softmax. Сумма весов равна `1` **до применения dropout**.





Готовлю общий датасет для валидации. Беру все предложения из validation и привязываю эталонные BIO-метки CoNLL-2003 к первым частям исходных токенов. На этом датасете буду считать micro-F1 для обеих моделей

In [ ]:
validation_examples = []

for record in validation:
    encoded = tokenizer(record["tokens"], is_split_into_words=True)
    gold_tags = [tag_names[tag_id] for tag_id in record["ner_tags"]]

    validation_examples.append({
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": align_bio_labels(encoded.word_ids(), gold_tags),
    })

validation_dataset = Dataset.from_list(validation_examples)

print("Записей для валидации:", len(validation_dataset))

Записей для валидации: 3250


Загружаю предобученный DistilBERT с линейным слоем для девяти BIO-меток. Беру два предложения с разметкой GPT и проверяю, что модель принимает подготовленные данные, выдаёт логиты и считает потерю по переданным меткам

In [ ]:
import torch
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    student_checkpoint,
    num_labels=len(tag_names),
    id2label=dict(enumerate(tag_names)),
    label2id=tag_to_id,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
batch = data_collator([gpt_train_dataset[0], gpt_train_dataset[1]])

model.eval()
with torch.no_grad():
    output = model(**batch)


print("Размер входа:", tuple(batch["input_ids"].shape))
print("Размер логитов:", tuple(output.logits.shape))
print("Потеря:", output.loss.item())

model.safetensors: reconstructing file:   0%|          |  0.00B /  263MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Размер входа: (2, 21)
Размер логитов: (2, 21, 9)
Потеря: 2.416903018951416


Collator берёт готовые примеры, дополняет их до общей длины и собирает порцию

Режим оценки — это режим, в котором модель использует текущие веса, чтобы выдавать предсказания так, как будет делать это после обучения. В DistilBERT отключается dropout. Поэтому одно и то же предложение при тех же весах даёт предсказания без случайных изменений из-за dropout

Такой режим нужен на валидации, чтобы сравнить качество моделей в одинаковых условиях. В пробной ячейке включаем его, чтобы посмотреть логиты и loss на двух предложениях из train. Режим зависит от того, что хотим сделать с моделью

DistilBERT с линейным слоем модель выдаёт девять логитов на каждой позиции. Затем через argmax берём индекс самого большого логита и по tag_names получаем название метки. Например, если максимум стоит на индексе 3, выбираем B-ORG

Если исходный токен CoNLL разбился на несколько подтокенов, для его метки используем логиты первого подтокена. Softmax для выбора можно пропустить: у самого большого логита будет и самая большая вероятность

# Как обучается DistilBERT на BIO-метках



## 1. Какие веса загружаю перед обучением

`AutoModelForTokenClassification.from_pretrained(...)` загружает предобученный DistilBERT и создаёт модель для классификации токенов (уже с линейным слоем). Веса таблиц эмбеддингов, внимания и FFN уже получены при предобучении. Линейный слой для нашей задачи новый. Его веса инициализируются перед обучением, а число выходов задаёт `num_labels=len(tag_names)`. У нас девять выходов — по одному для каждой BIO-метки.

`id2label` связывает номер класса с названием метки, например `3 → B-ORG`. `label2id` задаёт обратное соответствие `B-ORG → 3`. Эти словари сообщают модели, что означают номера классов.

При полном дообучении можно обновлять параметры всего DistilBERT вместе с новым линейным слоем.

## 2. Что подаю модели

Из `gpt_train_dataset` беру две записи. Каждая содержит три поля:

| Поле | Что в нём лежит |
|---|---|
| `input_ids` | Коды токенов и подтокенов, по которым модель берёт векторы из таблицы эмбеддингов |
| `attention_mask` | Маска внимания. `1` отмечает обычные позиции входа, `0` — позиции заполнения `[PAD]` |
| `labels` | Целевые BIO-метки в виде числовых кодов. `-100` отмечает позиции, которые нужно пропустить при расчёте потери |

**Batch** — порция примеров, которые модель обрабатывает вместе. `DataCollatorForTokenClassification` собирает записи в одну порцию, выравнивает их по длине и превращает списки чисел в тензоры PyTorch. Вход дополняется токенами `[PAD]`, в маску на этих позициях ставится `0`, а в метках ставится `-100`.

Если самое длинное из двух предложений после токенизации занимает 12 позиций, `batch["input_ids"]` имеет размер `(2, 12)`. Две строки соответствуют двум предложениям, в каждой по 12 позиций.

Запись `model(**batch)` передаёт поля словаря в модель по именам. Для нашего набора это равнозначно такому вызову:

```python
model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    labels=batch["labels"],
)
```

## 3. Что такое прямой проход

**Прямой проход** — это вычисление от входа модели до её результата:

Токенизатор подготавливает коды токенов, DistilBERT обрабатывает их и выдаёт итоговый вектор из 768 чисел для каждой позиции. Затем линейный слой превращает этот вектор в 9 логитов — по одной оценке для каждой BIO-метки.

У линейного слоя есть матрица из 9 строк по 768 весов и 9 смещений. Каждая строка отвечает за одну метку. Чтобы получить её логит, слой умножает каждую из 768 координат входного вектора на соответствующий вес, складывает произведения и прибавляет смещение. Для следующей метки он использует тот же входной вектор, свою строку весов и своё смещение.

Например, пусть итоговый вектор токена `Hamburg` для простоты состоит из трёх чисел `[2, 5, 1]`. Тогда два логита могут рассчитаться так:

```text
B-ORG: 2 × 0.5 + 5 × (−0.2) + 1 × 0.7 + 0.1 = 0.8
O:     2 × 0.2 + 5 × 0.1    + 1 × 0.0 − 0.3 = 0.6
```

Если модели переданы целевые метки, в прямом проходе она также вычисляет `loss`. Для этого кросс-энтропия использует softmax, который переводит девять логитов в девять вероятностей с суммой 1.

Например, если для простоты оставить только две метки с логитами `0.8` для `B-ORG` и `0.6` для `O`, то вероятность `B-ORG` будет:

$$
p(\text{B-ORG}) = \frac{e^{0.8}}{e^{0.8} + e^{0.6}} \approx 0.55
$$

Если в `labels` для этого токена стоит `B-ORG`, потеря для позиции равна $-\ln(0.55) \approx 0.60$. Для всей порции усредняю потери по позициям с целевыми метками.

Softmax входит в вычисление кросс-энтропии. На вход функции потерь передаю логиты, и в `output.logits` модель тоже возвращает исходные оценки. В `output.loss` сохраняет число, которое вычислила кросс-энтропия. Если целевая метка — B-ORG, то из логитов [0.8, 0.6] получается вероятность около 0.55, а из неё — потеря около 0.60

Так за один проход для каждой позиции получается один набор из девяти логитов. В настоящем классификаторе обучаются все $9 \times 768 = 6912$ весов и 9 смещений. Слой использует эти параметры для всех токенов, а входной вектор из 768 чисел DistilBERT вычисляет для конкретного токена в его контексте. При полном дообучении обновляются и параметры самого DistilBERT.

Позиции с `-100` пропускаются. Например, если в первом предложении учитываются 2 токена, а во втором — 3, складываю 5 потерь и делю на 5.
loss считаем по текущей порции предложений (batch)

На этом прямой проход заканчивается. В пробной ячейке вывожу полученные логиты и `loss`. При обучении после прямого прохода PyTorch вычислит градиенты по `loss`, и оптимизатор обновит веса.



**Логиты** — оценки классов до softmax. Для порции из двух предложений по 12 позиций размер результата будет `(2, 12, 9)`. Последняя размерность означает, что на каждой позиции модель выдала девять оценок.


В пробной ячейке вызываю `model.eval()`, чтобы отключить dropout. `torch.no_grad()` говорит PyTorch пропустить запись вычислений, нужных для последующего вычисления градиентов. Поэтому в этой ячейке получаю `logits` и `loss`, а веса остаются прежними.



## 4. Как модель узнаёт, в какую сторону менять веса

**Градиент** — набор производных потери по обучаемым параметрам. Для одного веса производная показывает, как изменится потеря при небольшом увеличении этого веса, если остальные параметры оставить прежними.

- Положительная производная означает, что небольшое увеличение веса повысит потерю. Чтобы уменьшить потерю, можно сделать шаг в сторону уменьшения веса
- Отрицательная производная означает, что небольшое увеличение веса снизит потерю. Тогда можно сделать шаг в сторону увеличения веса

Чтобы получить градиенты, PyTorch выполняет **обратное распространение**. Он идёт от вычисленной потери назад по операциям прямого прохода и применяет правило дифференцирования сложной функции. Так градиент доходит сначала до линейного классификатора, а затем до обучаемых параметров DistilBERT, в том числе до матриц внимания и таблиц эмбеддингов.


## 5. Как градиент превращается в обновление веса

**Оптимизатор** — алгоритм, который берёт градиенты и обновляет параметры. Для понимания возьму обычный градиентный спуск. Он вычитает из веса градиент, умноженный на скорость обучения:

$$
w_{\text{новый}} = w_{\text{старый}} - \eta \frac{\partial L}{\partial w}
$$

$\eta$ — **скорость обучения**, или `learning rate`. Она задаёт масштаб шага. Например, если старый вес равен `1`, градиент равен `−4`, а скорость обучения равна `0.1`, получится:

```text
Новый вес = 1 − 0.1 × (−4) = 1.4
```

При обучении DistilBERT можно использовать **AdamW**. Он учитывает текущие и прошлые градиенты и подстраивает величину обновления для каждого параметра. Градиенты вычисляет PyTorch, а `optimizer.step()` выполняет само обновление. Этот процесс повторяется на разных порциях данных, поэтому потеря может колебаться от порции к порции.

## Как AdamW обновляет веса модели

AdamW — оптимизатор. После прямого прохода модель возвращает `loss` для текущей порции. PyTorch вычисляет производную этой потери по каждому обучаемому параметру.

`loss.backward()` вычисляет градиенты, а `optimizer.step()` использует их, чтобы изменить значения параметров. При запуске через Trainer эти операции выполняются внутри цикла обучения

### Какие числа AdamW хранит для каждого веса

Допустим, на трёх последовательных шагах обучения градиент одного и того же веса был −3, −5 и −4. Каждый градиент вычислен по потере на своей порции предложений. После каждого шага AdamW обновляет вес и сохраняет информацию о градиенте для следующих шагов

- **Среднее градиентов** сохраняет информацию о направлении. В примере градиенты отрицательные, поэтому их среднее тоже отрицательное. Если градиенты часто меняют знак, их вклады могут частично компенсировать друг друга
- **Среднее квадратов градиентов** отражает величину градиентов независимо от знака. В примере квадраты равны `9`, `25` и `16`. Корень из среднего квадратов служит оценкой типичного масштаба градиента

Свежие значения получают больший вес, чем старые. Например, при `beta1 = 0.9` AdamW берёт 90% прежнего среднего и добавляет 10% текущего градиента. Для среднего квадратов при `beta2 = 0.999` берёт 99.9% прежнего среднего и добавляет 0.1% квадрата текущего градиента. Поэтому среднее квадратов меняется медленнее

Оба средних в начале равны нулю. На первых шагах AdamW поправляет их значения, чтобы начальные нули не занижали оценку направления и масштаба

### Как получается шаг обновления

AdamW делит скорректированное среднее градиентов на корень из скорректированного среднего квадратов. К знаменателю прибавляет маленькое положительное число `epsilon`, чтобы избежать деления на ноль. Затем умножает результат на скорость обучения и вычитает из старого веса

$$
w_{\text{новый}}
=
w_{\text{старый}}
-
\eta\frac{\widehat{m}}{\sqrt{\widehat{v}}+\varepsilon}
$$

Здесь $w$ — один параметр модели, $\eta$ — скорость обучения, $\widehat{m}$ — скорректированное среднее градиентов, а $\widehat{v}$ — скорректированное среднее квадратов градиентов. В этой формуле пока опускаю weight decay




### Зачем AdamW поправляет средние на начальных шагах

AdamW хранит для каждого веса два числа. $m_t$ — среднее градиентов после шага $t$, а $v_t$ — среднее квадратов градиентов. До первого шага оба числа равны нулю

$\beta_1=0.9$ означает, что при обновлении среднего градиентов AdamW берёт 90% прежнего среднего и добавляет 10% нового градиента. $\beta_2=0.999$ работает так же для среднего квадратов: берём 99.9% прежнего среднего и добавляем 0.1% квадрата нового градиента

Допустим, первый градиент равен `−4`. Тогда среднее градиентов после первого шага равно `0.9 × 0 + 0.1 × (−4) = −0.4`. Из всего среднего только 10% приходится на реальный градиент, а остальные 90% — на начальный ноль. Чтобы учесть только реальный градиент, делим `−0.4` на `0.1` и получаем `−4`

На втором шаге доля начального нуля снова умножается на `0.9` и становится `0.9² = 0.81`. Значит, на два реальных градиента вместе приходится `1 − 0.81 = 0.19`. Теперь делим среднее на `0.19`

Поэтому после $t$ шагов делитель для среднего градиентов равен $1-\beta_1^t$. Со средним квадратов поступаем так же, используя $\beta_2$:

$$
\widehat{m}_t = \frac{m_t}{1-\beta_1^t},
\qquad
\widehat{v}_t = \frac{v_t}{1-\beta_2^t}
$$

$\widehat{m}_t$ и $\widehat{v}_t$ — средние после поправки. Именно их AdamW использует для обновления веса. С каждым шагом доля начального нуля уменьшается, поэтому делители постепенно приближаются к единице

## 6. Что будет делать Trainer

`Trainer` — готовый класс библиотеки `transformers`, который управляет циклом обучения. Передаем ему модель, обучающий набор, `data_collator` и настройки. Когда вызовем `trainer.train()`, он будет повторять такой цикл:

1. Берёт очередную порцию предложений из обучающего набора
2. Собирает входные тензоры и запускает прямой проход в режиме обучения
3. Получает `loss` и вычисляет градиенты
4. В нужный момент делает шаг оптимизатора
5. Переходит к следующей порции

**Эпоха** — один проход по обучающему набору. Например, при 97 предложениях и размере порции 8 за эпоху получится 13 порций, если сохранять последнюю неполную. Обучение может включать несколько эпох.

Для сравнения запущу обучение дважды. Каждый раз начну с одного и того же предобученного чекпоинта и одинаковых настроек. Предложения будут те же, а в `labels` один раз будут метки GPT, другой — эталонные метки.

### Как расшифровывается AdamW и что такое weight decay

Adam — сокращение от Adaptive Moment Estimation («адаптивная оценка моментов»). Здесь под моментами имеются в виду скользящее среднее градиентов и скользящее среднее их квадратов. По ним оптимизатор подстраивает величину шага для каждого параметра

W в названии AdamW означает Weight Decay. Полное название метода — Adam with decoupled weight decay («Adam с отделённым затуханием весов»)

Weight decay — дополнительное изменение веса в сторону нуля на каждом шаге оптимизатора. Для этой части обновления вес $w$ умножается на $1-\eta\lambda$, где $\eta$ — скорость обучения, а $\lambda$ — значение `weight_decay`. Например, если $w=0.5$ и множитель равен $0.99$, эта часть обновления изменит вес с `0.5` до `0.495`

Слово decoupled (из названия статьи) означает, что этот вклад считается отдельно от шага по градиенту. Поэтому итоговый вес может и вырасти, если градиент увеличивает его сильнее, чем weight decay уменьшает. При `weight_decay=0` эта часть обновления исчезает

Weight decay помогает модели меньше подстраиваться под случайные особенности обучающих примеров. Это один из способов регуляризации.

Представь, что в наших 97 предложениях слова в верхнем регистре часто оказываются названиями организаций. Модель может слишком сильно опереться на этот признак и начать ставить B-ORG другим словам в верхнем регистре. Weight decay слегка тянет обучаемые веса к нулю. Если сильный вес действительно помогает уменьшать ошибку, шаг по градиенту может поддерживать его или увеличивать. Если градиент почти не поддерживает этот вес, он постепенно уменьшается.

## 7. Где здесь валидация и micro-F1

`Loss` на обучении нужен, чтобы вычислять градиенты и обновлять веса. `Micro-F1` на валидации нужен, чтобы оценить, насколько хорошо модель находит **целые сущности с правильными границами и типом**.


In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./distilbert_gpt_pilot",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    optim="adamw_torch",
    logging_strategy="steps",
    logging_steps=1,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=gpt_train_dataset,
    data_collator=data_collator,
)

train_result = trainer.train()
print("Train loss:", train_result.metrics["train_loss"])

Step,Training Loss
1,2.328809
2,2.199388
3,2.059123
4,1.981712
5,1.892217
6,1.813371
7,1.746142
8,1.647044
9,1.718928
10,1.613252


Train loss: 1.7961735725402832


output_dir="./distilbert_gpt_pilot"	Задаёт папку запуска. Здесь Trainer мог бы сохранять чекпоинты, если включить их сохранение

num_train_epochs=1	Делаю одну эпоху, то есть один полный проход по 97 обучающим предложениям

per_device_train_batch_size=8	На одно устройство подаю до восьми предложений за раз. На одной GPU или CPU получу 12 порций по 8 предложений и последнюю порцию из одного предложения


learning_rate=2e-5	Задаю начальную скорость обучения 0.00002. Она задаёт масштаб обновлений весов. В ходе обучения расписание может менять её от шага к шагу

optim="adamw_torch"	Использую AdamW из PyTorch.

logging_strategy="steps"	Вывожу информацию об обучении через заданное число шагов оптимизатора

logging_steps=1	Вывожу информацию после каждого шага, поэтому увижу, как меняется loss от порции к порции

save_strategy="no"	Отключаю автоматическое сохранение чекпоинтов во время пробного запуска. Обновлённые веса остаются в объекте model в текущей сессии

report_to="none"	Отключаю отправку логов во внешние сервисы. Вывод обучения по-прежнему увижу в Colab

Расписание скорости обучения (scheduler) задаёт, какое значение learning_rate использовать на каждом шаге обновления весов.
У нас learning_rate=2e-5, то есть 0.00002. Это начальное значение. По умолчанию Trainer использует линейное расписание: в нашем запуске оно постепенно уменьшает скорость от 0.00002 к нулю за 13 шагов. Поэтому в конце обучения масштаб обновлений меньше, чем в начале. huggingface.co
Scheduler меняет общую скорость обучения по номеру шага. А AdamW использует эту скорость и историю градиентов, чтобы рассчитать, на сколько обновить каждый вес.

In [ ]:
def compute_ner_metrics(eval_prediction):
    predicted_ids = eval_prediction.predictions.argmax(axis=-1)
    gold_ids = eval_prediction.label_ids

    total_tp, total_fp, total_fn = 0, 0, 0

    for predicted_row, gold_row in zip(predicted_ids, gold_ids):
        predicted_tags = []
        gold_tags = []

        for predicted_id, gold_id in zip(predicted_row, gold_row):
            if gold_id == -100:
                continue

            predicted_tags.append(tag_names[predicted_id])
            gold_tags.append(tag_names[gold_id])

        tp, fp, fn = count_matches(gold_tags, predicted_tags)
        total_tp += tp
        total_fp += fp
        total_fn += fn

    precision, recall, micro_f1 = scores_from_counts(total_tp, total_fp, total_fn)

    return {
        "precision": precision,
        "recall": recall,
        "micro_f1": micro_f1,
    }

compute_ner_metrics — функция для оценки модели на размеченных данных. Она получает логиты модели и эталонные метки, сравнивает найденные сущности с эталоном и возвращает словарь с precision, recall и micro_f1.

axis=-1 означает «последняя ось». argmax(axis=-1) для каждой позиции просматривает её девять логитов и возвращает индекс самого большого числа, начиная отсчёт с нуля. Например, если максимум 1.4 стоит на индексе 5, результатом будет 5. В нашем tag_names это B-LOC.

Перехожу к разметке всего train для сравнения архитектур. Использую уже сохранённые ответы на первые 100 записей и запрашиваю разметку GPT для остальных. Как и в пробном запуске, один раз повторяю запрос для записей, где первый ответ не прошёл проверку

In [ ]:
from shutil import copyfile

full_first_path = runs_dir / "train_full_first.jsonl"
full_retry_path = runs_dir / "train_full_retry.jsonl"

if not full_first_path.exists():
    copyfile(train_first_path, full_first_path)

if not full_retry_path.exists() and train_retry_path.exists():
    copyfile(train_retry_path, full_retry_path)

for start in range(0, len(train_order), 500):
    batch_indices = train_order[start:start + 500]
    label_train_rows(batch_indices, full_first_path)
    print(f"Пройдено записей: {start + len(batch_indices)}/{len(train_order)}")

with full_first_path.open(encoding="utf-8") as file:
    full_first_results = [json.loads(line) for line in file]

full_retry_indices = [
    result["row_index"]
    for result in full_first_results
    if result["predicted_tags"] is None
]

print("Записей для повторного запроса:", len(full_retry_indices))
label_train_rows(full_retry_indices, full_retry_path)

Пройдено записей: 500/14041
Пройдено записей: 1000/14041
Пройдено записей: 1500/14041
Пройдено записей: 2000/14041
Пройдено записей: 2500/14041
Пройдено записей: 3000/14041
Пройдено записей: 3500/14041
Пройдено записей: 4000/14041
Пройдено записей: 4500/14041
Пройдено записей: 5000/14041
Пройдено записей: 5500/14041
Пройдено записей: 6000/14041
Пройдено записей: 6500/14041
Пройдено записей: 7000/14041
Пройдено записей: 7500/14041
Пройдено записей: 8000/14041
Пройдено записей: 8500/14041
Пройдено записей: 9000/14041
Пройдено записей: 9500/14041
Пройдено записей: 10000/14041
Пройдено записей: 10500/14041
Пройдено записей: 11000/14041
Пройдено записей: 11500/14041
Пройдено записей: 12000/14041
Пройдено записей: 12500/14041
Пройдено записей: 13000/14041
Пройдено записей: 13500/14041
Пройдено записей: 14000/14041
Пройдено записей: 14041/14041
Записей для повторного запроса: 1104


In [ ]:
from transformers import AutoModel, AutoTokenizer

backbone_checkpoints = {
    "distilbert": "distilbert/distilbert-base-cased",
    "bert": "google-bert/bert-base-cased",
}


def load_backbone(backbone_name):
    checkpoint = backbone_checkpoints[backbone_name]
    tokenizer = AutoTokenizer.from_pretrained(checkpoint, use_fast=True)
    backbone = AutoModel.from_pretrained(checkpoint)
    return tokenizer, backbone

load_backbone — принимает название бэкбона ("distilbert" или "bert"), находит его чекпоинт в backbone_checkpoints и возвращает два объекта: токенизатор и предобученный бэкбон. Токенизатор подготавливает входные данные, а бэкбон превращает их в контекстные векторы. Эти векторы дальше поступают в выбранный классификатор

Собираю общий обучающий набор из сохранённых ответов GPT. Для каждого предложения беру пригодный ответ первой попытки или повтора и добавляю эталонные метки CoNLL-2003. Если обе попытки не прошли проверку, пропускаю предложение для обоих вариантов обучения

In [ ]:
def read_tags_by_index(path):
    with path.open(encoding="utf-8") as file:
        results = (json.loads(line) for line in file)
        return {
            result["row_index"]: result["predicted_tags"]
            for result in results
        }


first_tags = read_tags_by_index(full_first_path)
retry_tags = read_tags_by_index(full_retry_path) if full_retry_path.exists() else {}

full_train_pairs = []

for row_index in train_order:
    gpt_tags = first_tags[row_index]

    if gpt_tags is None:
        gpt_tags = retry_tags[row_index]

    if gpt_tags is None:
        continue

    record = train[row_index]
    full_train_pairs.append({
        "row_index": row_index,
        "tokens": record["tokens"],
        "gpt_tags": gpt_tags,
        "gold_tags": [tag_names[tag_id] for tag_id in record["ner_tags"]],
    })

print(f"Предложений с двумя разметками: {len(full_train_pairs)}/{len(train_order)}")

Предложений с двумя разметками: 13560/14041


read_tags_by_index — принимает путь к JSONL-файлу с ответами GPT и возвращает словарь. Ключ — row_index, номер записи в train. Значение — список BIO-меток, если ответ прошёл проверку, или None, если проверка завершилась ошибкой. По этому словарю можно найти результат запроса для конкретного предложения

full_train_pairs — список словарей с предложениями, для которых удалось получить пригодные метки GPT с первой попытки или после повтора. В каждом словаре четыре поля:
- row_index — номер исходной записи в train
- tokens — список исходных токенов CoNLL
- gpt_tags — список BIO-меток из пригодного ответа GPT
- gold_tags — список эталонных BIO-меток CoNLL-2003 для тех же токенов

Из общего списка предложений собираю два датасета для обучения. В обоих одинаковые коды токенов и маски внимания. В поле labels одного датасета записываю метки GPT, в другом — эталонные метки CoNLL-2003

In [ ]:
def prepare_train_datasets(pairs, tokenizer):
    gpt_examples = []
    gold_examples = []

    for pair in pairs:
        encoded = tokenizer(pair["tokens"], is_split_into_words=True)
        word_ids = encoded.word_ids()

        gpt_examples.append({
            "input_ids": encoded["input_ids"],
            "attention_mask": encoded["attention_mask"],
            "labels": align_bio_labels(word_ids, pair["gpt_tags"]),
        })

        gold_examples.append({
            "input_ids": encoded["input_ids"],
            "attention_mask": encoded["attention_mask"],
            "labels": align_bio_labels(word_ids, pair["gold_tags"]),
        })

    return Dataset.from_list(gpt_examples), Dataset.from_list(gold_examples)


distil_gpt_train, distil_gold_train = prepare_train_datasets(
    full_train_pairs, tokenizer
)

print("Записей с метками GPT:", len(distil_gpt_train))
print("Записей с эталонными метками:", len(distil_gold_train))

Записей с метками GPT: 13560
Записей с эталонными метками: 13560


prepare_train_datasets — принимает список предложений с двумя версиями BIO-разметки (gpt_tags и gold_tags) и токенизатор выбранного бэкбона. Токенизирует каждое предложение и через align_bio_labels привязывает обе разметки к первым частям исходных токенов. Возвращает два датасета: первый с метками GPT, второй с эталонными метками CoNLL-2003. В обоих одинаковые input_ids и attention_mask, различается только поле labels

Запускаю линейный бейзлайн на полном обучающем наборе. Начинаю с предобученного чекпоинта и после каждой эпохи считаю micro-F1 на validation. Сохраняю веса с лучшим micro-F1, чтобы позже оценить выбранную модель на test

In [ ]:
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
    set_seed
)

def train_linear_baseline(
    checkpoint, tokenizer, train_data, eval_data, run_name, model_class=None
):
    set_seed(42)

    if model_class is None:
        model = AutoModelForTokenClassification.from_pretrained(
            checkpoint,
            num_labels=len(tag_names),
            id2label=dict(enumerate(tag_names)),
            label2id=tag_to_id,
        )
    else:
        model = model_class(checkpoint)

    args = TrainingArguments(
        output_dir=str(runs_dir / "linear_baselines" / run_name),
        num_train_epochs=3,
        per_device_train_batch_size=8,
        learning_rate=2e-5,
        optim="adamw_torch",
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="best",
        save_total_limit=1,
        save_only_model=True,
        load_best_model_at_end=True,
        metric_for_best_model="micro_f1",
        report_to="none",
        seed=42,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_data,
        eval_dataset=eval_data,
        data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
        compute_metrics=compute_ner_metrics,
    )

    trainer.train()
    return trainer.state.best_metric, trainer.state.best_model_checkpoint

train_linear_baseline — функция для дообучения моделей. Принимает идентификатор чекпоинта предобученной модели, токенизатор, обучающий и валидационный датасеты, имя запуска и необязательный аргумент model_class. По умолчанию создаёт модель с линейным слоем для BIO-меток. Если передан model_class, создаёт модель указанной конфигурации, например с BiLSTM, CRF или обоими компонентами

Функция передаёт созданную модель в Trainer. Во время обучения Trainer собирает примеры в порции и вызывает модель на каждой порции. При этом PyTorch запускает метод forward этой модели. После каждой эпохи Trainer оценивает модель на валидации с помощью compute_ner_metrics и сохраняет чекпоинт на Google Drive при улучшении micro-F1. Функция возвращает лучший micro-F1 и путь к соответствующему чекпоинту

| Настройка | Что делает |
|---|---|
| `logging_strategy="epoch"` | Выводит показатели обучения после каждой эпохи. За запуск в три эпохи получаем три значения `Training Loss` |
| `save_total_limit=1` | Ограничивает число сохраняемых чекпоинтов в папке этого запуска. Вместе с `save_strategy="best"` оставляем чекпоинт с лучшим micro-F1 |
| `save_only_model=True` | Сохраняет веса и конфигурацию модели. Состояние AdamW и расписания скорости обучения не сохраняется, поэтому чекпоинт подходит для оценки и предсказаний |
| `load_best_model_at_end=True` | В конце обучения загружает лучшие веса обратно в модель. В твоём запуске это веса **второй эпохи с F1 78,72%** |
| `report_to="none"` | Отключает отправку логов во внешние сервисы. Таблица в Colab всё равно выводится |
| `seed=42` | Задаёт начальное состояние случайных операций, которыми управляет `Trainer` |

От начальных весов линейного слоя зависят первые логиты, первая потеря и первые обновления весов. Если изменить seed, слой начнёт с других чисел. Даже на тех же предложениях обучение может пойти немного иначе, и итоговый micro-F1 может отличаться

In [ ]:
distil_gpt_f1, distil_gpt_checkpoint = train_linear_baseline(
    student_checkpoint,
    tokenizer,
    distil_gpt_train,
    validation_dataset,
    "distilbert_gpt_3ep",
)

print(f"Лучший micro-F1 на validation: {distil_gpt_f1:.2%}")
print("Чекпоинт:", distil_gpt_checkpoint)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Precision,Recall,Micro F1
1,0.176683,0.232499,0.747760,0.814709,0.779800
2,0.091256,0.262827,0.753229,0.824470,0.787241
3,0.063614,0.305941,0.751039,0.820936,0.784434


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Лучший micro-F1 на validation: 78.72%
Чекпоинт: /content/drive/MyDrive/llm-ner-conll2003/runs/linear_baselines/distilbert_gpt_3ep/checkpoint-3390


Дообучил DistilBERT с линейным слоем на 13 560 предложениях с GPT-метками. Лучший micro-F1 на validation получился после второй эпохи — 78,72%. На третьей эпохе train loss продолжил снижаться, а micro-F1 опустился до 78,44%, поэтому для сравнения беру чекпоинт второй эпохи

Перехожу к BERT-base

In [ ]:
bert_checkpoint = "google-bert/bert-base-cased"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_checkpoint, use_fast=True)

bert_gpt_train, bert_gold_train = prepare_train_datasets(
    full_train_pairs, bert_tokenizer
)

bert_validation_examples = []

for record in validation:
    encoded = bert_tokenizer(record["tokens"], is_split_into_words=True)
    gold_tags = [tag_names[tag_id] for tag_id in record["ner_tags"]]

    bert_validation_examples.append({
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": align_bio_labels(encoded.word_ids(), gold_tags),
    })

bert_validation_dataset = Dataset.from_list(bert_validation_examples)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Подготовил данные для BERT-base с его токенизатором. Для обучения взял те же 13 560 предложений, что и для DistilBERT, и собрал две версии с метками GPT и с эталонными метками. Для валидации использую эталонные метки CoNLL-2003

Начианаю обучать BERT-base с линейным слоем на метках GPT. Использую те же предложения и настройки обучения, что и в запуске DistilBERT на GPT-метках. Сравню лучший micro-F1 двух моделей на validation, чтобы выбрать бэкбон для экспериментов с BiLSTM и CRF

In [ ]:
bert_gpt_f1, bert_gpt_checkpoint = train_linear_baseline(
    bert_checkpoint,
    bert_tokenizer,
    bert_gpt_train,
    bert_validation_dataset,
    "bert_gpt",
)

print(f"Лучший micro-F1 на validation: {bert_gpt_f1:.2%}")
print("Чекпоинт:", bert_gpt_checkpoint)

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Precision,Recall,Micro F1
1,0.163163,0.235133,0.743687,0.827667,0.783433
2,0.083235,0.249027,0.757840,0.833726,0.793974
3,0.056360,0.296113,0.754260,0.834231,0.792233


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Лучший micro-F1 на validation: 79.40%
Чекпоинт: /content/drive/MyDrive/llm-ner-conll2003/runs/linear_baselines/bert_gpt/checkpoint-3390


Дообучил BERT-base с линейным слоем на 13 560 предложениях с метками GPT. Лучший micro-F1 на validation получился после второй эпохи — 79,40%. Сохранил чекпоинт checkpoint-3390. BERT-base опередил линейный DistilBERT на 0,67 процентного пункта, поэтому сначала проверю BiLSTM и CRF на BERT-base

Добавляю BiLSTM между BERT-base и линейным слоем. Из выхода BERT беру вектор первого подтокена каждого исходного токена. BiLSTM обрабатывает эту последовательность в двух направлениях, а линейный слой выдаёт девять логитов для каждого токена

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence,
    pad_sequence,
)
from transformers import AutoModel
from transformers.modeling_outputs import TokenClassifierOutput


class BiLSTMForTokenClassification(nn.Module):
    def __init__(self, checkpoint):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(checkpoint)
        self.bilstm = nn.LSTM(
            input_size=self.backbone.config.hidden_size,
            hidden_size=128,
            batch_first=True,
            bidirectional=True,
        )
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(256, len(tag_names))

    def forward(self, input_ids, attention_mask, labels=None, word_mask=None):
        if word_mask is None:
            if labels is None:
                raise ValueError("Передай labels или маску первых подтокенов")
            word_mask = labels.ne(-100)

        hidden_states = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
        ).last_hidden_state

        word_vectors = pad_sequence(
            [row[mask] for row, mask in zip(hidden_states, word_mask)],
            batch_first=True,
        )
        lengths = word_mask.sum(dim=1)
        packed = pack_padded_sequence(
            word_vectors, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_output, _ = self.bilstm(packed)
        word_output, _ = pad_packed_sequence(packed_output, batch_first=True)
        word_logits = self.classifier(self.dropout(word_output))

        word_steps = torch.arange(word_logits.size(1), device=word_logits.device)
        valid_words = word_steps[None, :] < lengths[:, None]
        selected_logits = word_logits[valid_words]

        logits = word_logits.new_zeros((*input_ids.shape, word_logits.size(-1)))
        logits[word_mask] = selected_logits

        loss = None
        if labels is not None:
            loss = F.cross_entropy(selected_logits, labels[word_mask])

        return TokenClassifierOutput(loss=loss, logits=logits)

`class BiLSTMForTokenClassification(nn.Module)` — класс модели для разметки исходных токенов CoNLL. Наследуется от `nn.Module`, базового класса моделей PyTorch. BERT вычисляет контекстные векторы подтокенов, BiLSTM обрабатывает последовательность векторов первых подтокенов, а линейный слой выдаёт по девять логитов BIO-меток на каждый исходный токен

`def __init__(self, checkpoint)` — метод инициализации модели. Вызывается при создании объекта класса. Принимает идентификатор чекпоинта, загружает предобученный бэкбон и создаёт BiLSTM, dropout и линейный слой. Сохраняет эти компоненты в объекте модели, чтобы использовать их при каждом прямом проходе

`def forward(self, input_ids, attention_mask, labels=None, word_mask=None)` — метод прямого прохода для одной порции предложений. Передаёт `input_ids` и `attention_mask` в BERT, по `word_mask` выбирает векторы первых подтокенов и пропускает их через BiLSTM, dropout и линейный слой. Если `word_mask` не передана, строит её по условию `labels != -100`. Возвращает логиты, разложенные по исходным позициям BERT, и потерю по кросс-энтропии, если переданы `labels`. Для предсказания без `labels` нужно передать `word_mask`, тогда `loss` будет равен `None`


Для примера. Пусть в порции 2 предложения, у каждого по 6 позиций на входе BERT с учётом служебных токенов и заполнения. После отбора первых подтокенов остаются 3 исходных токена в первом предложении и 2 во втором.

Например, форма (2, 6, 768) означает три оси — предложение, позиция, координата вектора. Ниже все формы указаны для этой порции.

| Переменная | Форма | Что внутри и зачем |
|---|---|---|
| `input_ids` | `(2, 6)` | Коды элементов словаря на каждой входной позиции. По ним BERT получает входные эмбеддинги. |
| `attention_mask` | `(2, 6)` | Единицы на реальных входных позициях и нули на `[PAD]`. BERT использует эту маску при вычислении внимания. |
| `labels` | `(2, 6)` | На первых подтокенах стоят ID целевых BIO-меток, на остальных позициях — `-100`. Может быть `None` при предсказании. |
| `word_mask` | `(2, 6)` | Булевы значения. `True` отмечает первый подтокен каждого исходного токена. Если маска не передана, код получает её через `labels != -100`. |
| `hidden_states` | `(2, 6, 768)` | Контекстные векторы после последнего слоя BERT. По одному вектору из 768 чисел на каждую входную позицию. |
| `[row[mask] for ...]` | Список тензоров `(3, 768)` и `(2, 768)` | Для каждого предложения из `hidden_states` выбраны только векторы первых подтокенов. |
| `word_vectors` | `(2, 3, 768)` | `pad_sequence` собирает выбранные векторы в один тензор. Ко второму предложению добавляется один нулевой вектор, чтобы у обоих было по 3 позиции. |
| `lengths` | `(2,)` | Тензор `[3, 2]`. Хранит количество реальных токенов в каждом предложении после отбора. |
| `packed` | `PackedSequence`, внутри `data` формы `(5, 768)` | Упакованные векторы пяти реальных токенов. Структура хранит, какие предложения ещё продолжаются на каждом шаге. Добавленный нулевой вектор в BiLSTM не попадает. |
| `packed_output` | `PackedSequence`, внутри `data` формы `(5, 256)` | Выходы BiLSTM в упакованном виде. На каждый реальный токен приходится 256 чисел — по 128 от каждого направления. |
| `word_output` | `(2, 3, 256)` | Выходы BiLSTM после распаковки. Снова есть отдельная ось предложений. В конце второго предложения снова добавлена позиция заполнения. |
| `word_logits` | `(2, 3, 9)` | Девять логитов на каждую позицию `word_output` после dropout и линейного слоя. Здесь ещё есть строка для заполнения. |
| `word_steps` | `(3,)` | Тензор `[0, 1, 2]` с номерами позиций после распаковки. Нужен для сравнения с реальной длиной предложения. |
| `valid_words` | `(2, 3)` | Булева маска `[[True, True, True], [True, True, False]]`. Отмечает реальные токены в `word_logits`. |
| `selected_logits` | `(5, 9)` | Пять строк логитов, выбранных через `word_logits[valid_words]`. Сначала три строки первого предложения, затем две второго. |
| `logits` | `(2, 6, 9)` | Сначала заполнен нулями. Затем `logits[word_mask] = selected_logits` записывает пять строк обратно на исходные позиции BERT. Благодаря этому логиты можно сопоставить с эталонными `labels` по предложениям и позициям. |
| `labels[word_mask]` | `(5,)` | Пять ID целевых BIO-меток в том же порядке, что и пять строк `selected_logits`. |
| `loss` | `()` | Одно число — средняя кросс-энтропия по пяти реальным токенам. Если `labels` не переданы, значение остаётся `None`. |

## Как работает BiLSTM после BERT или DistilBERT

LSTM расшифровывается как Long Short-Term Memory. Это рекуррентный слой, который обрабатывает последовательность по шагам. На каждом шаге он получает вектор текущего токена и два состояния, вычисленные на предыдущем шаге. Из них он вычисляет новые состояния и передаёт их дальше по последовательности.

BiLSTM расшифровывается как Bidirectional Long Short-Term Memory. Это две LSTM с отдельными обучаемыми параметрами. Одна обрабатывает предложение слева направо, вторая — справа налево. Для каждого токена их выходы соединяются в один вектор.

### 1. Что поступает на вход

Сначала бэкбон вычисляет контекстные векторы всех подтокенов. Затем `word_mask` оставляет по одному вектору на исходный токен CoNLL. В этой реализации берётся вектор первого подтокена. Для BERT-base и DistilBERT в эксперименте размер этого вектора равен 768.

Например, в предложении `New York won` в BiLSTM поступают три вектора:

```text
New   → x₁, 768 чисел
York  → x₂, 768 чисел
won   → x₃, 768 чисел
```

Здесь $x_t$ — уже вычисленный **контекстный вектор** исходного токена на позиции $t$. BiLSTM берёт эти векторы и дополнительно обрабатывает их как упорядоченную последовательность.

В коде `pack_padded_sequence` передаёт в LSTM только реальные позиции согласно `lengths`. Если в порции одно предложение содержит три исходных токена, а другое два, LSTM обрабатывает для них три и два шага соответственно. Каждое предложение имеет собственные состояния.

### 2. Что такое скрытое состояние и состояние ячейки

В одном направлении у LSTM на каждом шаге есть два вектора:

| Обозначение | Размер в этой модели | Что хранится |
|---|---:|---|
| $h_t$ | 128 | Скрытое состояние. Это выход LSTM на текущей позиции. Он также участвует в вычислениях на следующем шаге |
| $c_t$ | 128 | Состояние ячейки. Оно передаётся дальше и обновляется специальной формулой, которая объединяет прежнее состояние и новую информацию |

На шаге $t$ LSTM получает $x_t$ из 768 чисел, а также $h_{t-1}$ и $c_{t-1}$ по 128 чисел. Она вычисляет новые $h_t$ и $c_t$.

В вызове `self.bilstm(packed)` начальные состояния не передаются. Поэтому PyTorch начинает с нулевых $h_0$ и $c_0$ для каждого предложения в каждом направлении. У прямого направления первый шаг соответствует началу предложения, у обратного — его концу.

### 3. Как вычисляются вентили LSTM

На каждом шаге LSTM вычисляет три вектора вентилей и один вектор кандидата. **Вентиль** — набор коэффициентов, которыми LSTM поэлементно масштабирует другой вектор. В каждом из четырёх векторов по 128 координат.

| Обозначение | Название | Роль |
|---|---|---|
| $i_t$ | Входной вентиль, *input gate* | Задаёт вклад новой информации в состояние ячейки |
| $f_t$ | Вентиль забывания, *forget gate* | Задаёт, какая доля каждой координаты прежнего состояния сохраняется |
| $g_t$ | Кандидат для обновления | Содержит новые значения, которые можно добавить к состоянию ячейки |
| $o_t$ | Выходной вентиль, *output gate* | Регулирует вклад каждой координаты обновлённого состояния ячейки в скрытое состояние |

В формулах ниже $W^{(x)}$ умножается на текущий вход $x_t$, а $W^{(h)}$ — на предыдущее скрытое состояние $h_{t-1}$. У каждого из четырёх вычислений свой набор матриц и смещений.

$$
\begin{aligned}
i_t &= \sigma\!\left(W_i^{(x)}x_t + b_i^{(x)} + W_i^{(h)}h_{t-1} + b_i^{(h)}\right), \\
f_t &= \sigma\!\left(W_f^{(x)}x_t + b_f^{(x)} + W_f^{(h)}h_{t-1} + b_f^{(h)}\right), \\
g_t &= \tanh\!\left(W_g^{(x)}x_t + b_g^{(x)} + W_g^{(h)}h_{t-1} + b_g^{(h)}\right), \\
o_t &= \sigma\!\left(W_o^{(x)}x_t + b_o^{(x)} + W_o^{(h)}h_{t-1} + b_o^{(h)}\right).
\end{aligned}
$$

Для каждого вычисления $W^{(x)}$ имеет форму $128 \times 768$, а $W^{(h)}$ — $128 \times 128$. Например, чтобы получить первую координату входного вентиля, LSTM умножает все 768 координат $x_t$ на веса из первой строки $W_i^{(x)}$ и складывает произведения. Отдельно она так же обрабатывает 128 координат $h_{t-1}$ с первой строкой $W_i^{(h)}$. Затем складывает обе суммы, прибавляет два смещения и применяет сигмоиду. Для второй координаты использует вторые строки матриц, и так далее.

Сигмоида $\sigma(z) = 1/(1+e^{-z})$ переводит каждое входное число в интервал $(0,1)$. Коэффициент около нуля почти убирает соответствующий вклад, а коэффициент около единицы почти полностью его сохраняет. Функция $\tanh$ переводит каждое число в интервал $(-1,1)$, поэтому кандидат $g_t$ может добавлять как положительные, так и отрицательные значения.
$$
\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}
$$

**Выученные параметры** здесь — матрицы $W$ и смещения $b$. Они одни и те же на всех шагах одного направления. **Значения вентилей** вычисляются заново на каждом шаге, потому что меняются $x_t$ и $h_{t-1}$. Так одни и те же параметры могут давать разные коэффициенты для разных токенов.

### 4. Как обновляются состояния

Когда получены $i_t$, $f_t$, $g_t$ и $o_t$, LSTM вычисляет:

$$
c_t = f_t \odot c_{t-1} + i_t \odot g_t
$$

$$
h_t = o_t \odot \tanh(c_t)
$$

Символ $\odot$ означает **поэлементное умножение**. Первая координата умножается на первую, вторая на вторую и так далее. Например, $[2, 3] \odot [0.5, 0.2] = [1, 0.6]$.

В первой формуле есть два слагаемых. $f_t \odot c_{t-1}$ — часть прежнего состояния, которая сохраняется. $i_t \odot g_t$ — новая информация, которую LSTM добавляет на этом шаге. Затем $\tanh(c_t)$ и выходной вентиль определяют $h_t$.

**Числовой пример для одной из 128 координат.** Пусть ранее в этой координате состояния ячейки было $c_{t-1}=0.8$. После вычислений по формулам выше на текущем шаге получились $f_t=0.75$, $i_t=0.4$, $g_t=-0.5$ и $o_t=0.8$. Тогда:

$$
c_t = 0.75 \cdot 0.8 + 0.4 \cdot (-0.5)
    = 0.6 - 0.2
    = 0.4
$$

$$
h_t = 0.8 \cdot \tanh(0.4) \approx 0.304
$$

В этой координате сохранилось $0.6$ от прежнего состояния, а новая информация уменьшила его на $0.2$. На следующий шаг LSTM передаст и обновлённое состояние ячейки $0.4$, и скрытое состояние примерно $0.304$. Такой расчёт выполняется для всех 128 координат.

### 5. Как работают два направления

Для предложения `New York won` порядок обработки такой:

| Направление | Порядок обработки | Как вычисляется выход на позиции `York` |
|---|---|---|
| Прямое | `New → York → won` | Использует $x_{\text{York}}$ и состояния после обработки `New` |
| Обратное | `won → York → New` | Использует тот же $x_{\text{York}}$ и состояния после обработки `won` |

Оба направления работают со всеми токенами предложения. У двух направлений собственные матрицы, смещения и состояния. Их выходы PyTorch сопоставляет с исходными позициями токенов.

Для `York` получаются два скрытых состояния по 128 координат. Сначала в итоговом векторе идут 128 координат прямого направления, затем 128 координат обратного:

$$
y_t = [\,h_t^{\rightarrow};\ h_t^{\leftarrow}\,]
$$

То есть `hidden_size=128` задаёт размер **одного направления**, а `bidirectional=True` даёт по **256 выходных чисел на токен**. Вызов с `num_layers=1` по умолчанию означает один рекуррентный слой с двумя направлениями.

### 6. Что возвращает BiLSTM в этом коде

```python
packed_output, _ = self.bilstm(packed)
word_output, _ = pad_packed_sequence(packed_output, batch_first=True)
word_logits = self.classifier(self.dropout(word_output))
```

`packed_output` содержит выходные векторы по 256 чисел для всех реальных токенов. Второе возвращённое значение — пара финальных состояний `(h_n, c_n)` по направлениям.

`pad_packed_sequence` возвращает обычный тензор с осями «предложение, позиция, координата». Например, для двух предложений длины 3 и 2 его форма — `(2, 3, 256)`. После BiLSTM применяется внешний `Dropout(0.1)`, затем линейный слой переводит каждый вектор из 256 чисел в 9 логитов BIO-меток. Из расчёта потери позиции заполнения исключаются.

### 7. Как BiLSTM обучается

В варианте `BERT → BiLSTM → линейный слой` кросс-энтропия сравнивает логиты реальных токенов с целевыми BIO-метками. Во время обратного распространения PyTorch проходит от потери через линейный слой к BiLSTM, а затем к бэкбону, если его параметры не заморожены.

Градиент проходит и через связи между шагами LSTM. Например, потеря на позиции `York` зависит от прямого скрытого состояния на этой позиции, а оно зависит от состояний, вычисленных на `New`. Поэтому при расчёте градиентов учитывается и путь через предыдущий шаг. Это называется **обратным распространением ошибки во времени**. Оно выполняется для обоих направлений по их собственному порядку обработки.

AdamW обновляет обучаемые матрицы и смещения. При следующем прямом проходе LSTM использует обновлённые параметры и заново вычисляет состояния и значения вентилей для новой порции. Состояния $h_t$ и $c_t$ живут внутри прямого прохода. Между порциями сохраняются обучаемые параметры слоя.

Входные векторы BiLSTM уже содержат контекст, потому что это выходные векторы BERT или DistilBERT. BiLSTM добавляет поверх них ещё один способ обработки порядка токенов — передачу состояния вдоль предложения в двух направлениях. Улучшает ли это распознавание сущностей в данном эксперименте, проверяется сравнением micro-F1 с линейным бейзлайном на одной и той же валидационной выборке.

In [ ]:
bert_bilstm_f1, bert_bilstm_checkpoint = train_linear_baseline(
    bert_checkpoint,
    bert_tokenizer,
    bert_gpt_train,
    bert_validation_dataset,
    "bert_bilstm_gpt",
    model_class=BiLSTMForTokenClassification,
)

print(f"Лучший micro-F1 на validation: {bert_bilstm_f1:.2%}")
print("Чекпоинт:", bert_bilstm_checkpoint)

### BERT-base + BiLSTM + линейный слой

Дообучил модель на 13 560 предложениях с GPT-разметкой.

| Эпоха | Training loss | Validation loss | Precision | Recall | Micro-F1 |
|---:|---:|---:|---:|---:|---:|
| 1 | 0.221519 | 0.225161 | 74.76% | 82.41% | 78.40% |
| 2 | 0.099396 | 0.230414 | 75.59% | 83.47% | 79.33% |
| 3 | 0.071282 | 0.255914 | 75.94% | 83.37% | **79.48%** |

Training loss снижался, а validation loss рос. Micro-F1 на валидации вырос с 78,40% до 79,48%. Лучший результат получен на третьей эпохе, поэтому сохранил `checkpoint-5085`. Добавление BiLSTM дало **+0,08 процентного пункта** относительно BERT-base с линейным слоем (79,40%).

Начинаю пробовать CRF

In [ ]:
%pip install -q pytorch-crf==0.7.2

In [ ]:
from functools import partial
from torchcrf import CRF


class CRFForTokenClassification(nn.Module):
    def __init__(self, checkpoint, use_bilstm=False):
        super().__init__()
        self.use_bilstm = use_bilstm

        if use_bilstm:
            self.scorer = BiLSTMForTokenClassification(checkpoint)
        else:
            self.scorer = AutoModelForTokenClassification.from_pretrained(
                checkpoint,
                num_labels=len(tag_names),
                id2label=dict(enumerate(tag_names)),
                label2id=tag_to_id,
            )

        self.crf = CRF(len(tag_names), batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None, word_mask=None):
        if word_mask is None:
            if labels is None:
                raise ValueError("Передай labels или маску первых подтокенов")
            word_mask = labels.ne(-100)

        scorer_inputs = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
        }
        if self.use_bilstm:
            scorer_inputs["word_mask"] = word_mask

        logits = self.scorer(**scorer_inputs).logits

        word_scores = pad_sequence(
            [row[keep] for row, keep in zip(logits, word_mask)],
            batch_first=True,
        ).float()

        lengths = word_mask.sum(dim=1)
        positions = torch.arange(word_scores.size(1), device=word_scores.device)
        crf_mask = positions[None, :] < lengths[:, None]

        loss = None
        if labels is not None:
            word_tags = pad_sequence(
                [row[keep] for row, keep in zip(labels, word_mask)],
                batch_first=True,
                padding_value=0,
            )
            loss = -self.crf(
                word_scores, word_tags, mask=crf_mask, reduction="token_mean"
            )

        if self.training:
            return TokenClassifierOutput(loss=loss, logits=logits)

        paths = self.crf.decode(word_scores, mask=crf_mask)
        tag_ids = torch.tensor(
            [tag for path in paths for tag in path],
            device=logits.device,
            dtype=torch.long,
        )

        decoded = torch.zeros_like(logits)
        decoded[word_mask] = F.one_hot(
            tag_ids, num_classes=len(tag_names)
        ).to(logits.dtype)

        return TokenClassifierOutput(loss=loss, logits=decoded)

### Класс CRFForTokenClassification

`class CRFForTokenClassification(nn.Module)` — класс модели для разметки токенов с помощью CRF. Вложенная модель вычисляет по девять оценок BIO-меток на токен. CRF использует эти оценки вместе с обучаемыми оценками переходов между соседними метками. При обучении он вычисляет потерю для эталонной последовательности, а при предсказании выбирает лучшую последовательность меток целиком

`def __init__(self, checkpoint, use_bilstm=False)` — метод инициализации модели. Принимает идентификатор чекпоинта и флаг `use_bilstm`. При `False` создаёт модель с бэкбоном и линейным слоем, при `True` — модель с бэкбоном, BiLSTM и линейным слоем. Сохраняет эту модель в `self.scorer` и создаёт `self.crf` для девяти BIO-меток. CRF хранит обучаемые оценки начала, конца и переходов между метками

`def forward(self, input_ids, attention_mask, labels=None, word_mask=None)` — метод прямого прохода для одной порции предложений. Получает оценки от `self.scorer` и по `word_mask` оставляет по одной строке оценок на исходный токен. Если маска не передана, строит её по условию `labels != -100`. Затем собирает оценки по предложениям и строит `crf_mask`, которая отмечает реальные токены после заполнения. Если переданы `labels`, вычисляет потерю CRF. В режиме обучения возвращает потерю и исходные логиты. В режиме оценки декодирует последовательность меток для каждого предложения и возвращает её в one-hot виде на исходных позициях BERT. Для предсказания без `labels` нужно передать `word_mask`, тогда `loss` будет равен `None`

`self.crf(word_scores, word_tags, mask=crf_mask, reduction="token_mean")` — вычисляет логарифм вероятности эталонной последовательности меток для каждого предложения, суммирует эти значения и делит на общее число реальных токенов порции. Минус перед вызовом превращает результат в потерю, которую минимизирует оптимизатор

`self.crf.decode(word_scores, mask=crf_mask)` — выбирает последовательность BIO-меток с наибольшей общей оценкой для каждого предложения с помощью алгоритма Витерби. Возвращает список, в котором для каждого предложения хранится отдельный список ID предсказанных меток



Пусть в порции два предложения, у каждого по шесть позиций на входе BERT с учётом служебных токенов и заполнения. После отбора первых подтокенов у первого предложения остаётся три исходных токена, у второго — два. Всего пять токенов. Число BIO-меток равно девяти.

| Переменная | Форма | Что внутри и зачем |
|---|---|---|
| `input_ids` | `(2, 6)` | ID элементов словаря на входных позициях. По ним бэкбон получает входные эмбеддинги. |
| `attention_mask` | `(2, 6)` | Единицы на входных позициях и нули на `[PAD]`. Бэкбон использует маску при вычислении внимания. |
| `labels` | `(2, 6)` | ID эталонных BIO-меток на первых подтокенах и `-100` на остальных позициях. Используется для построения `word_mask` и расчёта потери. Может быть `None` при предсказании. |
| `word_mask` | `(2, 6)` | Булева маска исходных позиций BERT. `True` отмечает первый подтокен исходного токена. По этой маске выбираются оценки и целевые метки для CRF. Она же задаёт позиции для записи предсказаний. |
| `scorer_inputs` | Словарь | Входы для `self.scorer`. Содержит `input_ids` и `attention_mask`, а в варианте с BiLSTM ещё и `word_mask`. |
| `logits` | `(2, 6, 9)` | Исходные оценки вложенной модели. На каждой позиции BERT есть девять логитов. Из этого тензора выбираются строки для CRF. |
| `word_scores` | `(2, 3, 9)` | Выбранные логиты после `pad_sequence`. Во второе предложение добавляется строка из девяти нулей. |
| `lengths` | `(2,)` | Тензор `[3, 2]`. Хранит число исходных токенов в каждом предложении. |
| `positions` | `(3,)` | Тензор `[0, 1, 2]`. Содержит номера позиций в `word_scores` для сравнения с реальной длиной предложения. |
| `crf_mask` | `(2, 3)` | Булева маска `[[True, True, True], [True, True, False]]`. Показывает CRF, какие позиции учитывать. Последняя позиция второго предложения — заполнение. |
| `word_tags` | `(2, 3)` | ID эталонных меток в том же порядке, что и `word_scores`. Например, `[[3, 0, 5], [1, 0, 0]]`. Последний `0` во втором предложении добавлен для заполнения и пропускается по `crf_mask`. |
| `paths` | Список списков длины 3 и 2 | Результат декодирования на оценке. Например, `[[3, 0, 5], [1, 0]]`. Для каждого предложения хранится своя последовательность предсказанных ID. |
| `tag_ids` | `(5,)` | Все предсказанные ID в порядке предложений и токенов. В примере `[3, 0, 5, 1, 0]`. Нужен для `F.one_hot`. |
| `F.one_hot(tag_ids, num_classes=len(tag_names))` | `(5, 9)` | По одной строке на предсказанный токен. На позиции выбранной метки стоит `1`, на остальных — `0`. Например, для ID `3` единица стоит на позиции `3`. |
| `decoded[word_mask]` | `(5, 9)` | Пять мест внутри `decoded`, куда записываются one-hot строки. Их порядок совпадает с порядком ID в `tag_ids`. |
| `decoded` | `(2, 6, 9)` | Тензор формы исходных логитов. На первых подтокенах записаны one-hot строки, на остальных позициях остаются нули. Возвращается под именем `logits`, чтобы функция метрик получила выбранные CRF метки через `argmax`. |

`word_mask` работает на исходной сетке BERT `(2, 6)`. Она выбирает первые подтокены и затем возвращает предсказания на эти же позиции. `crf_mask` работает на сетке исходных токенов после `pad_sequence` `(2, 3)`. Она нужна CRF, чтобы пропускать заполнение и обрабатывать каждое предложение как отдельную последовательность.

В режиме обучения `TokenClassifierOutput.logits` содержит исходные оценки классификатора. В режиме оценки это поле содержит `decoded`. Его единицы и нули кодируют метки, которые уже выбрал CRF.

## Как CRF выбирает разметку и считает потерю

У CRF в этой модели два расчёта. **Витерби выбирает ответ модели** — одну последовательность BIO-меток с наибольшей оценкой. **Потеря CRF показывает, какую вероятность модель присвоила эталонной последовательности** среди всех возможных разметок предложения.

В обоих расчётах на первом токене получается 9 чисел. На каждом следующем токене из этих 9 чисел строится 81 кандидат, по 9 на каждую текущую метку. Затем для каждой текущей метки её 9 кандидатов снова объединяются в одно число. Витерби берёт максимум, а при расчёте потери используется `logsumexp`.

### 1. Витерби — как модель выбирает ответ

**Что на входе**

- `word_scores` — по 9 сырых логитов на каждый исходный токен. Один логит — оценка одной BIO-метки на конкретном токене.
- Параметры CRF — оценки начала предложения, переходов между соседними метками и конца предложения.
- `crf_mask` — маска реальных токенов. Предложение заканчивается на последней позиции с `True`.

**Что считаем на первом токене**

Для каждой из девяти меток складываем **её логит на первом токене** и **оценку начала предложения с этой метки**. Получаем 9 чисел. Каждое число — оценка пути длиной в один токен.

Например, для первого токена оценка метки `O` — это логит `O` плюс `start_transitions` для `O`. Аналогично вычисляются оценки `B-ORG`, `I-ORG` и остальных меток.

**Что считаем на втором токене**

Фиксируем одну возможную метку второго токена, например `B-ORG`. На первом токене есть девять возможных меток, поэтому для `B-ORG` на втором токене получаем девять кандидатов:

| Метка первого токена | Кандидат для `B-ORG` на втором токене |
|---|---|
| `O` | Накопленная оценка для `O` + переход `O → B-ORG` + логит `B-ORG` на втором токене |
| `B-ORG` | Накопленная оценка для `B-ORG` + переход `B-ORG → B-ORG` + тот же логит `B-ORG` |
| Остальные семь меток | Накопленная оценка каждой метки + её переход в `B-ORG` + тот же логит `B-ORG` |

Из этих девяти кандидатов берём **максимум**. Сохраняем две вещи — значение максимума и метку первого токена, из которой он получился.

Повторяем расчёт для каждой из девяти меток второго токена. Всего рассматриваем **81 кандидата**. После отбора остаются **9 оценок лучших путей** и **9 указателей на предыдущую метку**.

**Какой смысл у этих девяти оценок**

Например, число для `B-ORG` означает «оценка лучшей частичной разметки от начала предложения до текущего токена при условии, что текущий токен помечен как `B-ORG`». Число для `O` означает то же самое для путей, заканчивающихся на `O`.

Сохраняем отдельную оценку для каждой метки, потому что на следующем токене переходы из `O` и из `B-ORG` могут иметь разные оценки. Если два пути уже заканчиваются на одной и той же метке, при одинаковом продолжении они получат одинаковые добавки. Поэтому для этой метки достаточно сохранить лучший из них.

**Что считаем дальше и что получаем в итоге**

На третьем и каждом следующем токене повторяем **9 → 81 → 9**. К оценке лучшего пути с предыдущего шага прибавляем переход и логит текущей метки. Для каждой текущей метки снова выбираем максимум из девяти кандидатов и запоминаем, откуда он пришёл.

После последнего реального токена к каждой из девяти оценок прибавляем оценку конца предложения с соответствующей меткой. Из получившихся девяти чисел выбираем **один максимум**. Его метка становится последней меткой ответа. Затем идём по сохранённым указателям справа налево и восстанавливаем все остальные метки.

В коде `self.crf.decode(word_scores, mask=crf_mask)` возвращает **один список предсказанных ID меток на каждое предложение**. Например, `[[3, 0, 5], [1, 0]]` означает три метки для первого предложения и две для второго.

### 2. Потеря CRF — как оценивается эталонная разметка

**Что на входе**

Поступают те же логиты, параметры переходов и маска. Дополнительно `word_tags` содержит **эталонные ID меток**. Для потери нужно вычислить две величины:

1. **Оценку эталонной последовательности.** Путь уже задан в `word_tags`. Складываем оценку начала, логиты эталонных меток, оценки переходов между ними и оценку конца.
2. **Общий вес всех возможных последовательностей.** Для каждой последовательности её вес равен `exp(общая оценка)`. Нужно учесть сумму этих весов. CRF вычисляет её в логарифмах, сохраняя по одному накопленному числу для каждой последней метки.

**Что считаем на первом токене**

Начинаем так же, как в Витерби. Для каждой метки складываем её логит и оценку начала. Получаем **9 чисел**.

Здесь каждое число можно понимать как **логарифм веса** одного варианта. Например, если логит `B-ORG` плюс её оценка начала равны `0.8`, вес этого варианта равен `exp(0.8)`, а логарифм веса — `0.8`.

**Что считаем на втором токене**

Для каждой метки второго токена строим 9 кандидатов, по одному на каждую метку первого токена:

```text
кандидат = старый логарифм веса + оценка перехода + логит текущей метки
```

Всего получаем **81 кандидат**. Дальше для каждой текущей метки отдельно выполняем три действия:

1. Возводим `e` в степень каждого из её девяти кандидатов. Получаем девять весов.
2. Складываем эти девять весов.
3. Берём натуральный логарифм суммы.

Эта операция называется **`logsumexp`**. После неё снова остаются **9 чисел**, по одному на каждую метку второго токена.

**Какой смысл у девяти чисел и что получаем в итоге**

Например, число для `I-ORG` теперь означает «логарифм суммарного веса всех частичных разметок от начала предложения до текущего токена, которые заканчиваются на `I-ORG`». Каждый следующий шаг учитывает всё больше вариантов разметки от начала предложения до текущего токена, а на хранение по-прежнему идут девять накопленных чисел.

После последнего реального токена прибавляем к каждому из девяти чисел оценку конца для его метки. Затем объединяем эти девять чисел через `logsumexp`. Получается **$\ln Z$**, где $Z$ — сумма весов всех возможных разметок предложения. При девяти метках и $T$ реальных токенах она учитывает все $9^T$ разметок.



| Расчёт | Как объединяются 9 кандидатов для текущей метки | Что получаем |
|---|---|---|
| Витерби | Максимум и указатель на предыдущую метку | Лучшую последовательность BIO-меток для каждого предложения |
| Потеря CRF | `logsumexp`, который учитывает вклад каждого кандидата | Логарифм общего веса всех разметок для расчёта `loss` по эталонным меткам |

In [ ]:
bert_base = "google-bert/bert-base-cased"

bert_crf_f1, bert_crf_checkpoint = train_linear_baseline(
    bert_base,
    bert_tokenizer,
    bert_gpt_train,
    bert_validation_dataset,
    "bert_crf_gpt",
    model_class=CRFForTokenClassification,
)

print(f"BERT + CRF: {bert_crf_f1:.2%}")
print(bert_crf_checkpoint)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Precision,Recall,Micro F1
1,0.162907,0.231831,0.743880,0.828509,0.783917
2,0.082802,0.248784,0.758726,0.834063,0.794613
3,0.055968,0.294951,0.757400,0.835409,0.794494


BERT + CRF: 79.46%
/content/drive/MyDrive/llm-ner-conll2003/runs/linear_baselines/bert_crf_gpt/checkpoint-3390


Дообучил BERT-base с линейным слоем и CRF на 13 560 предложениях с GPT-разметкой. Лучший micro-F1 на валидации составил 79,46% на второй эпохе — на 0,06 процентного пункта выше линейного бейзлайна BERT-base (79,40%). Сохранил веса в checkpoint-3390.

In [ ]:
bert_bilstm_crf_f1, bert_bilstm_crf_checkpoint = train_linear_baseline(
    bert_base,
    bert_tokenizer,
    bert_gpt_train,
    bert_validation_dataset,
    "bert_bilstm_crf_gpt",
    model_class=partial(CRFForTokenClassification, use_bilstm=True),
)

print(f"BERT + BiLSTM + CRF: {bert_bilstm_crf_f1:.2%}")
print(bert_bilstm_crf_checkpoint)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Precision,Recall,Micro F1
1,0.222352,0.224142,0.751377,0.826489,0.787145


### BERT-base + BiLSTM + линейный слой + CRF

Дообучил модель на 13 560 предложениях с GPT-разметкой. В выводе ячейки сохранилась только первая эпоха. Остальные значения восстановил из `trainer_state.json`.

| Эпоха | Training loss | Validation loss | Precision | Recall | Micro-F1 |
|---:|---:|---:|---:|---:|---:|
| 1 | 0.222352 | 0.224142 | 75.14% | 82.65% | 78.71% |
| 2 | 0.099675 | 0.230574 | 75.65% | 83.29% | 79.29% |
| 3 | 0.071401 | 0.252248 | 76.48% | 83.47% | **79.83%** |

Micro-F1 на валидации рос после каждой эпохи, хотя validation loss тоже увеличивался. Лучший чекпоинт по micro-F1 — `checkpoint-5085` после третьей эпохи. Добавление CRF к BERT-base с BiLSTM дало **+0,35 процентного пункта** на валидации: с 79,48% до 79,83%.

In [ ]:
distil_bilstm_crf_f1, distil_bilstm_crf_checkpoint = train_linear_baseline(
    student_checkpoint,
    tokenizer,
    distil_gpt_train,
    validation_dataset,
    "distilbert_bilstm_crf_gpt",
    model_class=partial(CRFForTokenClassification, use_bilstm=True),
)

print(f"DistilBERT + BiLSTM + CRF: {distil_bilstm_crf_f1:.2%}")
print(distil_bilstm_crf_checkpoint)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Precision,Recall,Micro F1
1,0.243610,0.225100,0.738026,0.811680,0.773103
2,0.108403,0.231445,0.752835,0.826826,0.788098
3,0.078451,0.255259,0.750268,0.823124,0.785009


DistilBERT + BiLSTM + CRF: 78.81%
/content/drive/MyDrive/llm-ner-conll2003/runs/linear_baselines/distilbert_bilstm_crf_gpt/checkpoint-3390


Дообучил DistilBERT с BiLSTM, линейным слоем и CRF на 13 560 предложениях с GPT-разметкой. Лучший micro-F1 на валидации составил **78,81%** на второй эпохе. Сохранил веса в `checkpoint-3390`. BERT-base с той же головой набрал 79,83%, на 1,02 процентного пункта выше. Поэтому для обучения на эталонных метках выбрал BERT-base.

In [ ]:
bert_bilstm_crf_gold_f1, bert_bilstm_crf_gold_checkpoint = train_linear_baseline(
    "google-bert/bert-base-cased",
    bert_tokenizer,
    bert_gold_train,
    bert_validation_dataset,
    "bert_bilstm_crf_gold",
    model_class=partial(CRFForTokenClassification, use_bilstm=True),
)

print(f"BERT + BiLSTM + CRF, эталон: {bert_bilstm_crf_gold_f1:.2%}")
print(bert_bilstm_crf_gold_checkpoint)

### BERT-base + BiLSTM + CRF на эталонной разметке

Дообучил модель на эталонных метках тех же 13 560 предложений, которые ранее использовал для обучения на GPT-разметке.

| Эпоха | Training loss | Validation loss | Precision | Recall | Micro-F1 |
|---:|---:|---:|---:|---:|---:|
| 1 | 0.155843 | 0.052504 | 92.83% | 93.98% | 93.40% |
| 2 | 0.037074 | 0.043672 | 94.10% | 94.95% | 94.52% |
| 3 | 0.020236 | 0.042898 | 94.69% | 95.42% | **95.05%** |

Лучший чекпоинт — `bert_bilstm_crf_gold/checkpoint-5085`.

**Вывод.** За три эпохи снизились и обучающая, и валидационная потери. Precision, recall и micro-F1 на валидации росли после каждой эпохи. Лучший micro-F1 получен на третьей эпохе. На валидации версия на эталонных метках набрала 95,05%, а та же архитектура на GPT-метках — 79,83%. Разница по исходным значениям составила 15,23 процентного пункта. Дальше сравню оба сохранённых чекпоинта на одном тестовом наборе.

In [ ]:
bert_test_examples = []

for record in test:
    encoded = bert_tokenizer(record["tokens"], is_split_into_words=True)
    gold_tags = [tag_names[tag_id] for tag_id in record["ner_tags"]]

    bert_test_examples.append({
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": align_bio_labels(encoded.word_ids(), gold_tags),
    })

bert_test_dataset = Dataset.from_list(bert_test_examples)
print("Предложений в test:", len(bert_test_dataset))

Предложений в test: 3453


Подготовил 3 453 предложения из `test` CoNLL-2003. Токенизировал их тем же токенизатором BERT-base, что и валидацию, и привязал эталонные BIO-метки к первым подтокенам. На этом наборе оценю обе версии модели — обученную на GPT-метках и обученную на эталонных метках.

In [ ]:
import pandas as pd
from safetensors.torch import load_file


def predict_bert_crf_on_test(checkpoint_dir):
    weights_path = checkpoint_dir / "model.safetensors"
    if not weights_path.is_file():
        raise FileNotFoundError(f"Не найден файл весов: {weights_path}")

    model = CRFForTokenClassification(bert_checkpoint, use_bilstm=True)
    model.load_state_dict(load_file(str(weights_path)), strict=True)

    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir="/content/bert_crf_test_eval",
            per_device_eval_batch_size=8,
            report_to="none",
        ),
        data_collator=DataCollatorForTokenClassification(
            tokenizer=bert_tokenizer,
        ),
        compute_metrics=compute_ner_metrics,
    )

    return trainer.predict(bert_test_dataset)


checkpoint_root = runs_dir / "linear_baselines"
checkpoint_dirs = {
    "GPT": checkpoint_root / "bert_bilstm_crf_gpt" / "checkpoint-5085",
    "Эталон": checkpoint_root / "bert_bilstm_crf_gold" / "checkpoint-5085",
}

test_outputs = {}
for source, checkpoint_dir in checkpoint_dirs.items():
    print(f"Оцениваю модель на разметке «{source}»")
    test_outputs[source] = predict_bert_crf_on_test(checkpoint_dir)

test_results = pd.DataFrame.from_dict(
    {
        source: {
            "Precision": output.metrics["test_precision"],
            "Recall": output.metrics["test_recall"],
            "Micro-F1": output.metrics["test_micro_f1"],
        }
        for source, output in test_outputs.items()
    },
    orient="index",
)
test_results.index.name = "Разметка для обучения"

display(test_results.style.format("{:.2%}"))

f1_gap_pp = 100 * (
    test_results.loc["Эталон", "Micro-F1"]
    - test_results.loc["GPT", "Micro-F1"]
)
print(f"Разница в test micro-F1 (эталон − GPT): {f1_gap_pp:+.2f} п.п.")

Оцениваю модель на разметке «GPT»


model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Оцениваю модель на разметке «Эталон»


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,Precision,Recall,Micro-F1
Разметка для обучения,,,
GPT,72.78%,80.61%,76.50%
Эталон,90.03%,91.66%,90.84%


Разница в test micro-F1 (эталон − GPT): +14.35 п.п.


predict_bert_crf_on_test — функция оценки сохранённой модели BERT-base с BiLSTM, линейным слоем и CRF на тестовом наборе CoNLL-2003. Принимает путь к папке чекпоинта, создаёт модель той же архитектуры и загружает её веса из model.safetensors.

`load_state_dict(..., strict=True)` загружает веса по именам параметров. Например, тензор `crf.transitions` из файла записывается в матрицу переходов CRF. Если в файле отсутствует ожидаемый параметр или есть лишний, PyTorch выдаёт ошибку.

Затем вызывает Trainer.predict() на bert_test_dataset. CRF выбирает последовательность BIO-меток для каждого предложения, а compute_ner_metrics сравнивает предсказанные сущности с эталонными по точному совпадению границ и типа. Функция возвращает предсказания, эталонные метки и метрики на тесте, включая precision, recall и micro-F1.

По готовым предсказаниям на test посчитаю precision, recall и F1 отдельно для PER, ORG, LOC и MISC. Сравню обе версии модели по каждому типу, чтобы понять, где разница между обучением на GPT-метках и на эталоне заметнее

In [ ]:
from collections import Counter

entity_types = ("PER", "ORG", "LOC", "MISC")
rows_by_type = []
errors_by_model = {}

for source, output in test_outputs.items():
    predicted_ids = output.predictions.argmax(axis=-1)

    correct_by_type = Counter()
    predicted_by_type = Counter()
    gold_by_type = Counter()
    errors_by_model[source] = []

    for sentence_id, (predicted_row, gold_row) in enumerate(
        zip(predicted_ids, output.label_ids)
    ):
        keep = gold_row != -100
        predicted_tags = [tag_names[tag_id] for tag_id in predicted_row[keep]]
        gold_tags = [tag_names[tag_id] for tag_id in gold_row[keep]]

        predicted_spans = set(bio_to_spans(predicted_tags))
        gold_spans = set(bio_to_spans(gold_tags))

        correct_by_type.update(span[2] for span in predicted_spans & gold_spans)
        predicted_by_type.update(span[2] for span in predicted_spans)
        gold_by_type.update(span[2] for span in gold_spans)

        if predicted_spans != gold_spans:
            errors_by_model[source].append((
                sentence_id,
                sorted(gold_spans - predicted_spans),
                sorted(predicted_spans - gold_spans),
            ))

    for entity_type in entity_types:
        tp = correct_by_type[entity_type]
        fp = predicted_by_type[entity_type] - tp
        fn = gold_by_type[entity_type] - tp
        precision, recall, f1 = scores_from_counts(tp, fp, fn)

        rows_by_type.append({
            "Разметка для обучения": source,
            "Тип": entity_type,
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "Сущностей в эталоне": gold_by_type[entity_type],
        })

test_by_type = pd.DataFrame(rows_by_type).set_index(
    ["Тип", "Разметка для обучения"]
).sort_index()

display(test_by_type.style.format({
    "Precision": "{:.2%}",
    "Recall": "{:.2%}",
    "F1": "{:.2%}",
}))

Главная разница — в распознавании организаций. У модели на GPT-метках recall для ORG составил 58,58% против 89,89% у эталонной версии. Это означает, что она гораздо реже находила организации с правильными границами и типом. Для LOC картина другая: recall почти одинаковый (92,93% и 93,53%), а precision у GPT-версии заметно ниже (67,77% против 92,47%). Значит, среди её предсказаний LOC было много лишних. Для MISC особенно низким тоже оказался precision (47,99%). Для PER качество двух версий оказалось близким.

Гипотеза: в обучающей GPT-разметке ORG могли чаще оставаться без метки или получать другой тип, а LOC и MISC - присваиваться фрагментам, которые в эталоне имеют другой тип или не выделены

Чтобы проверить, нужно сопоставить GPT-метки и эталон на одних и тех же тренировочных предложениях.

In [ ]:
def format_spans(tokens, spans):
    formatted_spans = []

    for start, end, entity_type in spans:
        entity_tokens = tokens[start:end]
        entity_text = " ".join(entity_tokens)
        formatted_span = f"{entity_text} [{entity_type}]"
        formatted_spans.append(formatted_span)

    if not formatted_spans:
        return "—"

    return ", ".join(formatted_spans)


for source, errors in errors_by_model.items():
    print(f"\n=== Модель на разметке «{source}» ===")

    for sentence_id, missed, extra in errors[:3]:
        tokens = test[sentence_id]["tokens"]

        print(f"\nПредложение test[{sentence_id}]")
        print("Текст:", " ".join(tokens))
        print("Что модель пропустила:", format_spans(tokens, missed))
        print("Что модель выделила лишнего:", format_spans(tokens, extra))


=== Модель на разметке «GPT» ===

Предложение test[0]
Текст: SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRISE DEFEAT .
Что модель пропустила: JAPAN [LOC]
Что модель выделила лишнего: JAPAN [ORG], LUCKY [PER]

Предложение test[2]
Текст: AL-AIN , United Arab Emirates 1996-12-06
Что модель пропустила: AL-AIN [LOC]
Что модель выделила лишнего: AL-AIN [ORG]

Предложение test[3]
Текст: Japan began the defence of their Asian Cup title with a lucky 2-1 win against Syria in a Group C championship match on Friday .
Что модель пропустила: —
Что модель выделила лишнего: Group C [MISC], Friday [MISC]

=== Модель на разметке «Эталон» ===

Предложение test[0]
Текст: SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRISE DEFEAT .
Что модель пропустила: JAPAN [LOC], CHINA [PER]
Что модель выделила лишнего: JAPAN [PER], LUCKY [PER], CHINA [LOC]

Предложение test[3]
Текст: Japan began the defence of their Asian Cup title with a lucky 2-1 win against Syria in a Group C championship match on Friday .
Что модель

Эти примеры показывают два вида ошибок: неверный тип при правильных границах и выделение лишней сущности. Такие ошибки встречаются у обеих версий

### Анализ результатов по типам сущностей

**LOC.** У модели на GPT-метках полнота почти такая же, как у модели на эталоне — **92,93%** против **93,53%**. При этом точность заметно ниже — **67,77%** против **92,47%**. Из-за этого F1 составляет **78,38%** против **93,00%**. Значит, GPT-версия находит большинство эталонных локаций, но часто выдаёт лишние LOC

**MISC.** У GPT-версии точность **47,99%**, а полнота **73,08%**. У эталонной версии выше обе метрики — **74,81%** и **82,91%**. F1 составляет **57,93%** против **78,65%**. Значит, GPT-версия и чаще пропускает эталонные сущности MISC, и чаще выделяет их лишний раз. Особенно заметна проблема с точностью: меньше половины её предсказаний MISC совпадают с эталоном. У обеих моделей это тип с самым низким F1.

**ORG.** У GPT-версии точность составляет **74,39%**, а полнота — **58,58%**. У эталонной версии эти показатели равны **88,34%** и **89,89%**. F1 составляет **65,54%** против **89,11%**. Здесь самая заметная разница — в полноте. Когда GPT-версия предсказывает ORG, большинство предсказаний совпадает с эталоном, но значительную часть эталонных организаций она не находит с правильными границами и типом.

**PER.** У обеих моделей высокие точность и полнота. У GPT-версии это **95,29%** и **93,82%**, у эталонной — **96,68%** и **95,36%**. F1 тоже близок — **94,55%** и **96,01%**. Значит, обе модели находят большинство эталонных имён людей, а среди их предсказаний PER мало лишних.

**Итог.** Модель на эталонной разметке получила более высокий F1 по всем четырём типам, но характер разницы зависит от типа. Для LOC и MISC у GPT-версии особенно заметна низкая точность, то есть высокая доля лишних предсказаний. Для ORG основная проблема — низкая полнота, то есть пропуски. Для PER результаты близки по всем трём метрикам.

### Предположения о причинах разницы

Обе модели обучались на одних и тех же предложениях с одинаковой архитектурой. Различался источник обучающих меток. По результатам теста можно предположить, что ошибки GPT-разметки распределялись по типам неравномерно.

- **ORG.** У модели на GPT-метках особенно низкая полнота. Возможно, GPT чаще пропускал организации в обучающих предложениях, присваивал им другой тип или выделял их с другими границами.
- **LOC и MISC.** У модели на GPT-метках особенно низкая точность. Возможно, GPT чаще присваивал эти метки фрагментам, которые в эталонной разметке относятся к другому типу или вообще не считаются сущностями.
- **PER.** Результаты двух моделей близки по точности, полноте и F1. Видимо, GPT размечал имена людей ближе к эталону, чем сущности остальных типов.

Тестовые метрики показывают, какие ошибки чаще делает обученная модель. Чтобы проверить их связь с обучающей разметкой, можно сравнить GPT-метки и эталон на тех же тренировочных предложениях и посмотреть пропуски, лишние сущности и ошибки в типах и границах.